# All-in-One TRACE-CRC Reproducibility Pipeline

This is the canonical, standalone reproducibility workflow for the
TRACE-CRC COPA 2026 artifact. It contains parameter selection, frozen
test evaluation, robustness analysis, and paper-figure export in one
ordered notebook.

The workflow is deliberately ordered:

1. load or generate deterministic compact diffusion predictions;
2. tune every baseline family on the same five inference seeds using only the
   fixed 300-trajectory calibration split;
3. tune the TRACE-CRC structure using its existing certified search unchanged;
4. freeze one baseline configuration per method and the TRACE/ablation
   structure before reading any test residuals;
5. recalibrate every method separately within each inference seed and evaluate
   on the corresponding untouched 700-trajectory test split;
6. save tuning records, tables, arrays, and Matplotlib figures under an
   isolated output directory.

For baseline tuning, each seed's 300 calibration trajectories are divided into
210 tuning-training and 90 tuning-validation trajectories. Candidate metrics
are computed per seed and then aggregated; the five seeds are not pooled or
treated as 1,500 independent trajectories. Horizon methods are selected against
their marginal coverage target, while trajectory methods are selected against
their trajectory-coverage target. A candidate is feasible only when all five
seed-level validation targets are met; among feasible candidates, mean
validation AFR is minimized. If a method has no all-seed-feasible candidate,
the deterministic fallback minimizes maximum seed-level coverage shortfall,
then mean shortfall, and then mean AFR. The 700 test trajectories never
participate in parameter selection.

This is a fairness-matched within-study protocol, not an independent
inference-seed generalization experiment: the same five seed conditions provide
calibration-only tuning evidence and are subsequently evaluated on their
disjoint test trajectories. Using separate tuning and evaluation seeds would
give a stronger seed-generalization study, but would require applying that
change to TRACE-CRC as well. The present design follows the broader ML
recommendation to account for randomness and hyperparameter-selection
variability while preserving a genuinely held-out test set; see
[Bouthillier et al. (2021)](https://arxiv.org/abs/2103.03098),
[Cawley and Talbot (2010)](https://www.jmlr.org/papers/v11/cawley10a.html),
and [Eimer et al. (2023)](https://proceedings.mlr.press/v202/eimer23a.html).


In [ ]:
from pathlib import Path
import copy
import csv
import json
import math
import pickle
import random
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from sklearn.base import clone
from sklearn.linear_model import QuantileRegressor, Ridge
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm


CWD = Path.cwd()
PROJECT_ROOT = CWD if (CWD / "DiU").exists() else CWD.parent
DIU_DIR = PROJECT_ROOT / "DiU"
DATA_DIR = PROJECT_ROOT / "data" / "csi"

if str(DIU_DIR) not in sys.path:
    sys.path.insert(0, str(DIU_DIR))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from inference_helpers import (
    add_snr_noise,
    auto_regressive_inference,
    compute_frobenius_norm,
    denormalize_csi,
    get_best_device,
    load_dataset_files,
    load_diu_checkpoint,
    normalize_csi,
)
from conformal.core.bounds import hb_p_value
from conformal.core.concentration import romano_wolf_HB

device = get_best_device()
print("project root:", PROJECT_ROOT)
print("device:", device)


In [ ]:
# =========================================================
# EXPERIMENT SETTINGS, PAPER NAMES, AND FIVE-SEED TUNING
# =========================================================

dataset_files = [DATA_DIR / "CSI_Dataset_1000_diverse.pt"]
max_samples_per_file = None
max_samples_total = 1000

checkpoint_path = DIU_DIR / "models" / "models_900.pt"
config_path = DIU_DIR / "config.yml"

T_in = 10
T_out = 20
inference_snr = 20.0
sampling_steps = 20
batch_size = 16

INFERENCE_SEEDS = [
    20260724,
    20260824,
    20260924,
    20261024,
    20261124,
]

# The primary comparison uses one predeclared random trajectory-level
# split. TRACE robustness uses 10 additional predeclared split seeds.
CALIBRATION_FRACTION = 0.30
CALIBRATION_SIZE = int(round(CALIBRATION_FRACTION * max_samples_total))
PRIMARY_SPLIT_SEED = 314159
ROBUSTNESS_SPLIT_SEEDS = [
    1101,
    1102,
    1103,
    1104,
    1105,
    1106,
    1107,
    1108,
    1109,
    1110,
]

alpha = 0.10
delta_ltt = 0.10
eps = 1e-8

# The baseline search must finish successfully before test evaluation.
USE_FIVE_SEED_BASELINE_TUNING = True

# Frozen TRACE-CRC settings from the revision experiment.
trace_profile_fraction = 0.1
trace_cp_fraction_within_rest = 0.15
trace_num_groups = 2
trace_smooth_window = 3
trace_floor_ratio = 0.15

trace_ridge_alpha = 1.0

# Prespecified LTT family shared by every CRC method. The full family is
# included in one Holm/HB correction; the extension is not tested repeatedly.
lambda_grid_initial = np.linspace(0.70, 1.30, 10)
lambda_grid_expand_step = 0.10
lambda_grid_max_cap = 3.00
lambda_grid_extension = np.arange(
    1.40,
    lambda_grid_max_cap + eps,
    lambda_grid_expand_step,
)
lambda_grid_master = np.concatenate(
    [lambda_grid_initial, lambda_grid_extension]
)
# Backward-compatible name used by the independent parameter-search cell.
lambda_grid = lambda_grid_initial.copy()

use_all_calib_global_crc = False

#The five-seed search below replaces this dictionary before any test evaluation.
FROZEN_BASELINE_PARAMS = {
    "weighted_horizon_beta": 0.01,
    "weighted_trajectory_beta": 0.05,
    "enbpi_horizon": {"B": 50, "window": None, "center_method": "mean"},
    "enbpi_trajectory": {"B": 50, "window": 100, "center_method": "mean"},
    "aci_horizon_gamma": 0.02,
    "aci_trajectory_gamma": 0.05,
    "agaci_horizon": {
        "gammas": np.array([0.0, 0.005, 0.01, 0.02, 0.03, 0.05]),
        "eta": 0.5,
    },
    "agaci_trajectory": {
        "gammas": np.array([0.0, 0.005, 0.01, 0.02, 0.03, 0.05]),
        "eta": 0.5,
    },
    "copulacpts": {"split": 0.60, "epochs": 1200},
}

M_GLOBAL_RESIDUAL = "Global Residual Conformal"
M_HORIZON_CONFORMAL = "Horizon-wise Conformal"
M_MAX_SCORE = "Max-score Split Conformal"
M_RESIDUAL_QUANTILE = "Residual Quantile Conformal"
M_WEIGHTED_HORIZON = "Exponentially Weighted Horizon Conformal"
M_WEIGHTED_TRAJECTORY = "Exponentially Weighted Trajectory Conformal"
M_ENBPI_HORIZON = "EnbPI-Horizon"
M_ENBPI_TRAJECTORY = "EnbPI-Trajectory"
M_ACI_HORIZON = "ACI-Horizon"
M_ACI_TRAJECTORY = "ACI-Trajectory"
M_AGACI_HORIZON = "AgACI-Horizon"
M_AGACI_TRAJECTORY = "AgACI-Trajectory"
M_BONFERRONI = "Bonferroni Horizon Conformal"
M_SIDAK = "Sidak Horizon Conformal"
M_COPULACPTS = "CopulaCPTS"
M_GLOBAL_CRC = "Global-CRC"
M_HORIZON_PROFILE_CRC = "Horizon-Profile CRC"
M_TRAJECTORY_STRATIFIED_CRC = "Trajectory-Stratified CRC"
M_TRACE_CRC = "TRACE-CRC"

C_STANDARD = "Standard split-conformal"
C_TIME = "Time-series adaptive"
C_STRUCTURED = "Structured multi-step"
C_RISK = "Risk control / ablation"

METHOD_GROUPS = {
    C_STANDARD: [
        M_GLOBAL_RESIDUAL,
        M_HORIZON_CONFORMAL,
        M_MAX_SCORE,
        M_RESIDUAL_QUANTILE,
    ],
    C_TIME: [
        M_WEIGHTED_HORIZON,
        M_WEIGHTED_TRAJECTORY,
        M_ENBPI_HORIZON,
        M_ENBPI_TRAJECTORY,
        M_ACI_HORIZON,
        M_ACI_TRAJECTORY,
        M_AGACI_HORIZON,
        M_AGACI_TRAJECTORY,
    ],
    C_STRUCTURED: [
        M_BONFERRONI,
        M_SIDAK,
        M_COPULACPTS,
    ],
    C_RISK: [
        M_GLOBAL_CRC,
        M_HORIZON_PROFILE_CRC,
        M_TRAJECTORY_STRATIFIED_CRC,
        M_TRACE_CRC,
    ],
}
METHOD_ORDER = [
    method
    for category_methods in METHOD_GROUPS.values()
    for method in category_methods
]

DISPLAY_NAME = {
    M_GLOBAL_RESIDUAL: "Global Residual",
    M_HORIZON_CONFORMAL: "Horizon-wise",
    M_MAX_SCORE: "Max-score Split",
    M_RESIDUAL_QUANTILE: "Residual Quantile",
    M_WEIGHTED_HORIZON: "EW-Horizon",
    M_WEIGHTED_TRAJECTORY: "EW-Trajectory",
    M_ENBPI_HORIZON: "EnbPI-Horizon",
    M_ENBPI_TRAJECTORY: "EnbPI-Trajectory",
    M_ACI_HORIZON: "ACI-Horizon",
    M_ACI_TRAJECTORY: "ACI-Trajectory",
    M_AGACI_HORIZON: "AgACI-Horizon",
    M_AGACI_TRAJECTORY: "AgACI-Trajectory",
    M_BONFERRONI: "Bonferroni-CRC",
    M_SIDAK: "Sidak-CRC",
    M_COPULACPTS: "CopulaCPTS",
    M_GLOBAL_CRC: "Global-CRC",
    M_HORIZON_PROFILE_CRC: "Horizon-Profile CRC",
    M_TRAJECTORY_STRATIFIED_CRC: "Trajectory-Stratified CRC",
    M_TRACE_CRC: "TRACE-CRC",
}

# These are copied from the revision notebook.
METHOD_COLORS = {
    M_GLOBAL_RESIDUAL: "tab:blue",
    M_HORIZON_CONFORMAL: "tab:orange",
    M_MAX_SCORE: "tab:green",
    M_RESIDUAL_QUANTILE: "tab:red",
    M_WEIGHTED_HORIZON: "tab:pink",
    M_WEIGHTED_TRAJECTORY: "tab:cyan",
    M_ENBPI_HORIZON: "olive",
    M_ENBPI_TRAJECTORY: "darkolivegreen",
    M_ACI_HORIZON: "magenta",
    M_ACI_TRAJECTORY: "violet",
    M_AGACI_HORIZON: "gold",
    M_AGACI_TRAJECTORY: "darkgoldenrod",
    M_BONFERRONI: "gray",
    M_SIDAK: "silver",
    M_COPULACPTS: "teal",
    M_GLOBAL_CRC: "navy",
    M_HORIZON_PROFILE_CRC: "darkorange",
    M_TRAJECTORY_STRATIFIED_CRC: "darkgreen",
    M_TRACE_CRC: "darkred",
}

# Shared Figure 2 family colors used by Matplotlib.
FIGURE_2_COLOR_STYLE = {
    M_WEIGHTED_HORIZON: "pairEW",
    M_WEIGHTED_TRAJECTORY: "pairEW",
    M_ENBPI_HORIZON: "pairEnbPI",
    M_ENBPI_TRAJECTORY: "pairEnbPI",
    M_ACI_HORIZON: "pairACI",
    M_ACI_TRAJECTORY: "pairACI",
    M_AGACI_HORIZON: "pairAgACI",
    M_AGACI_TRAJECTORY: "pairAgACI",
}
FIGURE_2_PAIR_COLORS = {
    "pairEW": METHOD_COLORS[M_WEIGHTED_HORIZON],
    "pairEnbPI": METHOD_COLORS[M_ENBPI_HORIZON],
    "pairACI": METHOD_COLORS[M_ACI_HORIZON],
    "pairAgACI": METHOD_COLORS[M_AGACI_HORIZON],
}
FIGURE_2_METHOD_COLORS = {
    method: FIGURE_2_PAIR_COLORS[color_style]
    for method, color_style in FIGURE_2_COLOR_STYLE.items()
}
FIGURE_2_DASHED_METHODS = {
    M_WEIGHTED_TRAJECTORY,
    M_ENBPI_TRAJECTORY,
    M_ACI_TRAJECTORY,
    M_AGACI_TRAJECTORY,
}

METHOD_CATEGORY = {
    method: category
    for category, methods in METHOD_GROUPS.items()
    for method in methods
}

# Explicit paper exclusions requested for plots/tables.
EXCLUDED_PAPER_METHODS = [
    "TRACE-Conformal",
    "Rolling Trajectory Conformal",
    "Rolling Horizon Conformal",
]

REBUILD_SEED_CACHES = False

robustness_dir = DIU_DIR / "final_comparison_revision_all_in_one_reproducibility"
cache_dir = robustness_dir / "compact_seed_inference"
legacy_cache_dirs = []
table_dir = robustness_dir / "tables"
figure_dir = robustness_dir / "figures"
for directory in (robustness_dir, cache_dir, table_dir, figure_dir):
    directory.mkdir(parents=True, exist_ok=True)

print("output root:", robustness_dir)
print("inference seeds:", INFERENCE_SEEDS)
print("primary split seed:", PRIMARY_SPLIT_SEED)
print("TRACE robustness split seeds:", ROBUSTNESS_SPLIT_SEEDS)
print("calibration/test sizes:", CALIBRATION_SIZE, max_samples_total - CALIBRATION_SIZE)
print("paper methods:", len(METHOD_ORDER))


In [ ]:
# =========================================================
# DETERMINISTIC COMPACT DIFFUSION INFERENCE
# =========================================================

def set_diffusion_seed(seed):
    seed = int(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    if (
        hasattr(torch.backends, "mps")
        and torch.backends.mps.is_available()
        and hasattr(torch, "mps")
        and hasattr(torch.mps, "manual_seed")
    ):
        torch.mps.manual_seed(seed)

    torch.use_deterministic_algorithms(True, warn_only=True)
    if hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def project_relative_path(path):
    resolved = Path(path).resolve()
    try:
        return resolved.relative_to(PROJECT_ROOT.resolve()).as_posix()
    except ValueError:
        return resolved.name


def inference_metadata(seed):
    return {
        "seed": int(seed),
        "dataset_files": [
            project_relative_path(path)
            for path in dataset_files
        ],
        "max_samples_total": int(max_samples_total),
        "checkpoint_path": project_relative_path(checkpoint_path),
        "config_path": project_relative_path(config_path),
        "T_in": int(T_in),
        "T_out": int(T_out),
        "inference_snr": float(inference_snr),
        "sampling_steps": int(sampling_steps),
        "batch_size": int(batch_size),
    }


def seed_cache_path(seed):
    return cache_dir / f"compact_seed_{int(seed)}.npz"


def seed_cache_candidates(seed):
    filename = f"compact_seed_{int(seed)}.npz"
    return [seed_cache_path(seed)] + [
        directory / filename for directory in legacy_cache_dirs
    ]


def load_valid_seed_cache(seed):
    if REBUILD_SEED_CACHES:
        return None, None

    expected = json.dumps(inference_metadata(seed), sort_keys=True)
    for path in seed_cache_candidates(seed):
        if not path.exists():
            continue
        with np.load(path, allow_pickle=False) as cache:
            actual = str(cache["metadata_json"].item())
            if actual != expected:
                raise RuntimeError(
                    f"Seed cache metadata mismatch for {path}. "
                    "Set REBUILD_SEED_CACHES=True to regenerate it intentionally."
                )
            return (
                {
                    "fro_residual": cache["fro_residual"].astype(float),
                    "predicted_norm": cache["predicted_norm"].astype(float),
                    "nmse_db": cache["nmse_db"].astype(float),
                },
                path,
            )
    return None, None


def run_compact_inference(dataset, bundle, seed):
    set_diffusion_seed(seed)

    data = torch.from_numpy(np.asarray(dataset)).float()
    past_all = data[:, :T_in]
    target_all = data[:, T_in:T_in + T_out]

    residual_batches = []
    predicted_norm_batches = []
    nmse_num = torch.zeros(T_out, dtype=torch.float64)
    nmse_den = torch.zeros(T_out, dtype=torch.float64)

    for start in tqdm(
        range(0, len(data), batch_size),
        desc=f"seed {seed}",
        leave=False,
    ):
        end = min(start + batch_size, len(data))
        past = past_all[start:end].to(device)
        target = target_all[start:end]

        past_normalized = normalize_csi(past, bundle["normalization_state"])
        past_noisy = add_snr_noise(past_normalized, inference_snr)
        predicted_normalized = auto_regressive_inference(
            past_noisy,
            horizon=T_out,
            lstm=bundle["lstm"],
            unet=bundle["unet"],
            scheduler=bundle["scheduler"],
            sampling_steps=sampling_steps,
            eta=0.0,
            guidance_scale=1.0,
            noise_level=100.0,
            show_progress=False,
        )
        predicted = denormalize_csi(
            predicted_normalized,
            bundle["normalization_state"],
        ).cpu()

        residual_batches.append(
            compute_frobenius_norm(
                target - predicted,
                channel_dim=2,
            ).cpu().numpy()
        )
        predicted_norm_batches.append(
            compute_frobenius_norm(
                predicted,
                channel_dim=2,
            ).cpu().numpy()
        )

        nmse_num += torch.sum(
            (predicted.double() - target.double()) ** 2,
            dim=(0, 2, 3, 4),
        )
        nmse_den += torch.sum(
            target.double() ** 2,
            dim=(0, 2, 3, 4),
        )

    residual = np.concatenate(residual_batches, axis=0)
    predicted_norm = np.concatenate(predicted_norm_batches, axis=0)
    nmse_db = (
        10.0 * torch.log10(nmse_num / (nmse_den + 1e-12))
    ).cpu().numpy()

    return {
        "fro_residual": residual,
        "predicted_norm": predicted_norm,
        "nmse_db": nmse_db,
    }


In [ ]:
# =========================================================
# LOAD OR GENERATE COMPACT DATA FOR EACH SEED
# =========================================================

seed_compact_data = {}
seeds_to_generate = []

for seed in INFERENCE_SEEDS:
    cached, loaded_path = load_valid_seed_cache(seed)
    if cached is None:
        seeds_to_generate.append(seed)
    else:
        seed_compact_data[seed] = cached
        print(f"Loaded compact seed cache: {loaded_path}")

        # Keep the new artifact self-contained without modifying the source
        # cache. Existing validated caches are copied into the new output root.
        destination = seed_cache_path(seed)
        if Path(loaded_path).resolve() != destination.resolve():
            np.savez_compressed(
                destination,
                fro_residual=np.asarray(cached["fro_residual"], dtype=np.float32),
                predicted_norm=np.asarray(cached["predicted_norm"], dtype=np.float32),
                nmse_db=np.asarray(cached["nmse_db"], dtype=np.float32),
                metadata_json=json.dumps(inference_metadata(seed), sort_keys=True),
            )
            print(f"Copied compact seed cache to: {destination}")

if seeds_to_generate:
    print("Generating compact inference for seeds:", seeds_to_generate)
    X = load_dataset_files(
        dataset_files,
        max_samples_per_file=max_samples_per_file,
        max_samples_total=max_samples_total,
    )

    if X.shape[1] < T_in + T_out:
        raise ValueError("Dataset sequence length is shorter than T_in + T_out.")

    bundle = load_diu_checkpoint(
        checkpoint_path=checkpoint_path,
        config_path=config_path,
        device=device,
        image_size=X.shape[-1],
        data_channels=X.shape[2],
        use_ema=True,
    )

    for seed in seeds_to_generate:
        compact = run_compact_inference(X, bundle, seed)
        path = seed_cache_path(seed)
        np.savez_compressed(
            path,
            fro_residual=np.asarray(compact["fro_residual"], dtype=np.float32),
            predicted_norm=np.asarray(compact["predicted_norm"], dtype=np.float32),
            nmse_db=np.asarray(compact["nmse_db"], dtype=np.float32),
            metadata_json=json.dumps(inference_metadata(seed), sort_keys=True),
        )
        seed_compact_data[seed] = {
            key: np.asarray(value, dtype=float)
            for key, value in compact.items()
        }
        print(f"Saved compact seed cache: {path}")

reference_shape = seed_compact_data[INFERENCE_SEEDS[0]]["fro_residual"].shape
for seed, compact in seed_compact_data.items():
    if compact["fro_residual"].shape != reference_shape:
        raise RuntimeError(f"Residual shape mismatch for seed {seed}.")
    if compact["predicted_norm"].shape != reference_shape:
        raise RuntimeError(f"Predicted-norm shape mismatch for seed {seed}.")

print("compact data shape per seed:", reference_shape)


In [ ]:
# =========================================================
# PAPER BASELINES
# =========================================================

def conformal_order_stat(values, alpha_level):
    values = np.asarray(values, dtype=float).reshape(-1)
    n = len(values)
    alpha_level = float(np.clip(alpha_level, 1e-8, 1 - 1e-8))
    k = math.ceil((n + 1) * (1 - alpha_level))
    k = min(max(k, 1), n)
    return np.sort(values)[k - 1]


def weighted_quantile(values, weights, level):
    values = np.asarray(values, dtype=float).reshape(-1)
    weights = np.asarray(weights, dtype=float).reshape(-1)
    assert values.shape == weights.shape

    order = np.argsort(values)
    sorted_values = values[order]
    sorted_weights = weights[order]

    cdf = np.cumsum(sorted_weights)
    cdf = cdf / cdf[-1]

    idx = np.searchsorted(cdf, level, side="left")
    idx = min(idx, len(sorted_values) - 1)

    return sorted_values[idx]


def decayed_weights(n, beta):
    ages = np.arange(n - 1, -1, -1)
    weights = np.exp(-beta * ages)
    return weights / weights.sum()


def pinball_loss(y, q, tau):
    y = np.asarray(y, dtype=float)
    q = np.asarray(q, dtype=float)
    return tau * np.maximum(y - q, 0.0) + (1.0 - tau) * np.maximum(q - y, 0.0)


def run_cqr_style_residual_quantile_cp(res_cal, res_test, alpha_level):
    n_cal_local, T_local = res_cal.shape

    if n_cal_local >= 4:
        split = n_cal_local // 2
        train_res = res_cal[:split]
        cal_res = res_cal[split:]
    else:
        train_res = res_cal
        cal_res = res_cal

    horizon_feature = np.linspace(0.0, 1.0, T_local)

    X_train = np.tile(horizon_feature[None, :], (train_res.shape[0], 1)).reshape(-1, 1)
    y_train = train_res.reshape(-1)
    X_horizon = horizon_feature.reshape(-1, 1)

    if QuantileRegressor is not None:
        try:
            model = QuantileRegressor(
                quantile=1 - alpha_level,
                alpha=1e-4,
                solver="highs",
            )
        except TypeError:
            model = QuantileRegressor(
                quantile=1 - alpha_level,
                alpha=1e-4,
            )

        model.fit(X_train, y_train)
        pred_q_h = model.predict(X_horizon)
    else:
        pred_q_h = np.quantile(train_res, 1 - alpha_level, axis=0)

    pred_q_h = np.maximum(pred_q_h, eps)

    cal_excess_scores = np.max(cal_res - pred_q_h[None, :], axis=1)
    q_adjust = conformal_order_stat(cal_excess_scores, alpha_level)
    radius_h = np.maximum(pred_q_h + q_adjust, eps)

    return {
        "radius": np.tile(radius_h[None, :], (res_test.shape[0], 1)),
        "pred_q_h": pred_q_h,
        "q_adjust": q_adjust,
    }


def run_weighted_cp(res_cal, res_test, alpha_level, beta, mode):
    n_test_local, T_local = res_test.shape
    radius = np.zeros_like(res_test)

    if mode == "stepwise":
        histories = [list(res_cal[:, t]) for t in range(T_local)]

        for i in range(n_test_local):
            for t in range(T_local):
                hist = np.asarray(histories[t], dtype=float)
                weights = decayed_weights(len(hist), beta)
                radius[i, t] = weighted_quantile(hist, weights, 1 - alpha_level)
                histories[t].append(float(res_test[i, t]))

    elif mode == "trajectory":
        history = list(np.max(res_cal, axis=1))

        for i in range(n_test_local):
            hist = np.asarray(history, dtype=float)
            weights = decayed_weights(len(hist), beta)
            q_i = weighted_quantile(hist, weights, 1 - alpha_level)
            radius[i, :] = q_i
            history.append(float(np.max(res_test[i])))

    else:
        raise ValueError("mode must be 'stepwise' or 'trajectory'.")

    return radius


class EnbPIDefaultMeanRegressor:
    def fit(self, X, y):
        self.mean_ = float(np.mean(y))
        return self

    def predict(self, X):
        return np.full(len(X), self.mean_, dtype=float)


def enbpi_clone_model(model):
    if clone is not None:
        try:
            return clone(model)
        except Exception:
            pass

    return copy.deepcopy(model)


def enbpi_bootstrap_indices(n, B, rng):
    return rng.integers(0, n, size=(B, n))


def enbpi_time_features(n_train, n_test, horizon_idx=None, T_total=None):
    train_time = np.arange(n_train, dtype=float)
    test_time = n_train + np.arange(n_test, dtype=float)

    denom = max(n_train + n_test - 1, 1)

    train_time = train_time / denom
    test_time = test_time / denom

    if horizon_idx is None:
        return train_time[:, None], test_time[:, None]

    h = float(horizon_idx) / max(T_total - 1, 1)

    X_train = np.column_stack([train_time, np.full(n_train, h)])
    X_test = np.column_stack([test_time, np.full(n_test, h)])

    return X_train, X_test


def enbpi_scalar(
    X_train,
    y_train,
    X_test,
    y_test=None,
    base_model=None,
    alpha_level=0.10,
    B=50,
    window=None,
    online_update=True,
    center_method="oob_quantile",
    random_state=123,
):
    X_train = np.asarray(X_train, dtype=float)
    y_train = np.asarray(y_train, dtype=float).reshape(-1)
    X_test = np.asarray(X_test, dtype=float)

    if y_test is not None:
        y_test = np.asarray(y_test, dtype=float).reshape(-1)

    n_train = len(y_train)
    n_test_local = len(X_test)

    if base_model is None:
        base_model = Ridge(alpha=1.0) if Ridge is not None else EnbPIDefaultMeanRegressor()

    rng = np.random.default_rng(random_state)
    boot_idx = enbpi_bootstrap_indices(n_train, B, rng)

    train_pred = np.zeros((B, n_train), dtype=float)
    test_pred = np.zeros((B, n_test_local), dtype=float)
    in_bootstrap = np.zeros((B, n_train), dtype=bool)

    models = []

    for b in range(B):
        model_b = enbpi_clone_model(base_model)
        idx_b = boot_idx[b]

        model_b.fit(X_train[idx_b], y_train[idx_b])

        train_pred[b] = np.asarray(model_b.predict(X_train), dtype=float).reshape(-1)
        test_pred[b] = np.asarray(model_b.predict(X_test), dtype=float).reshape(-1)

        in_bootstrap[b, idx_b] = True
        models.append(model_b)

    oob_residuals = np.zeros(n_train, dtype=float)
    oob_test_pred_by_train_point = np.zeros((n_train, n_test_local), dtype=float)

    for i in range(n_train):
        oob_models = np.where(~in_bootstrap[:, i])[0]

        if len(oob_models) > 0:
            pred_i = train_pred[oob_models, i].mean()
            oob_test_pred_by_train_point[i] = test_pred[oob_models].mean(axis=0)
        else:
            pred_i = train_pred[:, i].mean()
            oob_test_pred_by_train_point[i] = test_pred.mean(axis=0)

        oob_residuals[i] = abs(y_train[i] - pred_i)

    if center_method == "oob_quantile":
        center_idx = int((1 - alpha_level) * n_train)
        center_idx = min(max(center_idx, 0), n_train - 1)
        centers = np.sort(oob_test_pred_by_train_point, axis=0)[center_idx]

    elif center_method == "mean":
        centers = test_pred.mean(axis=0)

    else:
        raise ValueError("center_method must be 'oob_quantile' or 'mean'.")

    lower = np.zeros(n_test_local, dtype=float)
    upper = np.zeros(n_test_local, dtype=float)
    widths = np.zeros(n_test_local, dtype=float)

    residual_history = list(oob_residuals)

    for i in range(n_test_local):
        hist = residual_history if window is None else residual_history[-window:]
        width_i = conformal_order_stat(hist, alpha_level)

        widths[i] = width_i
        lower[i] = centers[i] - width_i
        upper[i] = centers[i] + width_i

        if online_update and y_test is not None:
            residual_history.append(abs(y_test[i] - centers[i]))

    return {
        "lower": lower,
        "upper": upper,
        "center": centers,
        "width": widths,
        "models": models,
        "boot_idx": boot_idx,
        "oob_residuals": oob_residuals,
        "final_residual_history": np.asarray(residual_history, dtype=float),
        "train_pred": train_pred,
        "test_pred": test_pred,
    }


def run_enbpi_csi_stepwise(
    fro_res_cal,
    fro_res_test,
    base_model=None,
    alpha_level=0.10,
    B=50,
    window=None,
    online_update=True,
    center_method="oob_quantile",
    random_state=123,
):
    res_cal_local = np.asarray(fro_res_cal, dtype=float)
    res_test_local = np.asarray(fro_res_test, dtype=float)

    n_cal_local, T_local = res_cal_local.shape
    n_test_local = res_test_local.shape[0]

    radius = np.zeros_like(res_test_local)
    outputs_by_horizon = []

    for t in range(T_local):
        X_train_t, X_test_t = enbpi_time_features(
            n_train=n_cal_local,
            n_test=n_test_local,
            horizon_idx=t,
            T_total=T_local,
        )

        out_t = enbpi_scalar(
            X_train=X_train_t,
            y_train=res_cal_local[:, t],
            X_test=X_test_t,
            y_test=res_test_local[:, t],
            base_model=base_model,
            alpha_level=alpha_level,
            B=B,
            window=window,
            online_update=online_update,
            center_method=center_method,
            random_state=random_state + t,
        )

        radius[:, t] = np.maximum(out_t["upper"], 0.0)
        outputs_by_horizon.append(out_t)

    covered = res_test_local <= radius

    return {
        "radius": radius,
        "covered": covered,
        "outputs_by_horizon": outputs_by_horizon,
        "pointwise_coverage": covered.mean(),
        "sequence_coverage": np.all(covered, axis=1).mean(),
        "trajectory_failure": (~np.all(covered, axis=1)).mean(),
        "avg_radius": radius.mean(),
    }


def run_enbpi_csi_trajectory(
    fro_res_cal,
    fro_res_test,
    base_model=None,
    alpha_level=0.10,
    B=50,
    window=None,
    online_update=True,
    center_method="oob_quantile",
    random_state=456,
):
    res_cal_local = np.asarray(fro_res_cal, dtype=float)
    res_test_local = np.asarray(fro_res_test, dtype=float)

    n_cal_local, T_local = res_cal_local.shape
    n_test_local = res_test_local.shape[0]

    y_train = np.max(res_cal_local, axis=1)
    y_test = np.max(res_test_local, axis=1)

    X_train, X_test = enbpi_time_features(
        n_train=n_cal_local,
        n_test=n_test_local,
    )

    out = enbpi_scalar(
        X_train=X_train,
        y_train=y_train,
        X_test=X_test,
        y_test=y_test,
        base_model=base_model,
        alpha_level=alpha_level,
        B=B,
        window=window,
        online_update=online_update,
        center_method=center_method,
        random_state=random_state,
    )

    score_radius = np.maximum(out["upper"], 0.0)
    radius = np.tile(score_radius[:, None], (1, T_local))
    covered = res_test_local <= radius

    return {
        "radius": radius,
        "covered": covered,
        "scalar_output": out,
        "pointwise_coverage": covered.mean(),
        "sequence_coverage": np.all(covered, axis=1).mean(),
        "trajectory_failure": (~np.all(covered, axis=1)).mean(),
        "avg_radius": radius.mean(),
    }


def aci_quantile_zaffran(values, alpha_t):
    values = np.asarray(values, dtype=float).reshape(-1)

    if alpha_t >= 1:
        return 0.0

    if alpha_t <= 0:
        return np.inf

    return float(np.quantile(values, 1.0 - alpha_t))


def run_aci_stepwise(res_cal, res_test, alpha_target, gamma, update_history=True):
    res_cal = np.asarray(res_cal, dtype=float)
    res_test = np.asarray(res_test, dtype=float)

    n_test_local, T_local = res_test.shape

    alpha_t = np.full(T_local, alpha_target, dtype=float)
    alpha_history = np.zeros((n_test_local, T_local), dtype=float)
    err_history = np.zeros((n_test_local, T_local), dtype=float)
    radius = np.zeros_like(res_test)

    histories = [list(res_cal[:, t]) for t in range(T_local)]

    for i in range(n_test_local):
        for t in range(T_local):
            alpha_history[i, t] = alpha_t[t]
            radius[i, t] = aci_quantile_zaffran(histories[t], alpha_t[t])

            err = float(res_test[i, t] > radius[i, t])
            err_history[i, t] = err

            #alpha_t[t] = alpha_t[t] + gamma * (alpha_target - err)
            alpha_t[t] = alpha_t[t] + gamma * (alpha_target - err)
            alpha_t[t] = float(np.clip(alpha_t[t], 1e-6, 1 - 1e-6))


            if update_history:
                histories[t].append(float(res_test[i, t]))

    return {
        "radius": radius,
        "alpha_history": alpha_history,
        "err_history": err_history,
        "final_alpha": alpha_t,
    }


def run_aci_trajectory(res_cal, res_test, alpha_target, gamma, update_history=True):
    res_cal = np.asarray(res_cal, dtype=float)
    res_test = np.asarray(res_test, dtype=float)

    n_test_local, T_local = res_test.shape

    alpha_t = float(alpha_target)
    alpha_history = np.zeros(n_test_local, dtype=float)
    err_history = np.zeros(n_test_local, dtype=float)
    radius = np.zeros_like(res_test)

    history_scores = list(np.max(res_cal, axis=1))

    for i in range(n_test_local):
        alpha_history[i] = alpha_t

        q_i = aci_quantile_zaffran(history_scores, alpha_t)
        radius[i, :] = q_i

        err = float(np.any(res_test[i, :] > radius[i, :]))
        err_history[i] = err

        #alpha_t = alpha_t + gamma * (alpha_target - err)
        alpha_t = alpha_t + gamma * (alpha_target - err)
        alpha_t = float(np.clip(alpha_t, 1e-6, 1 - 1e-6))


        if update_history:
            history_scores.append(float(np.max(res_test[i, :])))

    return {
        "radius": radius,
        "alpha_history": alpha_history,
        "err_history": err_history,
        "final_alpha": alpha_t,
    }


def run_aci_experts_stepwise(res_cal, res_test, alpha_target, gammas, update_history=True):
    expert_outputs = []

    for gamma in gammas:
        out = run_aci_stepwise(
            res_cal=res_cal,
            res_test=res_test,
            alpha_target=alpha_target,
            gamma=float(gamma),
            update_history=update_history,
        )
        out["gamma"] = float(gamma)
        expert_outputs.append(out)

    expert_radii = np.stack([out["radius"] for out in expert_outputs], axis=0)
    expert_alpha_history = np.stack([out["alpha_history"] for out in expert_outputs], axis=0)

    return {
        "expert_outputs": expert_outputs,
        "expert_radii": expert_radii,
        "expert_alpha_history": expert_alpha_history,
        "gammas": np.asarray(gammas, dtype=float),
    }


def run_aci_experts_trajectory(res_cal, res_test, alpha_target, gammas, update_history=True):
    expert_outputs = []

    for gamma in gammas:
        out = run_aci_trajectory(
            res_cal=res_cal,
            res_test=res_test,
            alpha_target=alpha_target,
            gamma=float(gamma),
            update_history=update_history,
        )
        out["gamma"] = float(gamma)
        expert_outputs.append(out)

    expert_radii = np.stack([out["radius"] for out in expert_outputs], axis=0)
    expert_alpha_history = np.stack([out["alpha_history"] for out in expert_outputs], axis=0)

    return {
        "expert_outputs": expert_outputs,
        "expert_radii": expert_radii,
        "expert_alpha_history": expert_alpha_history,
        "gammas": np.asarray(gammas, dtype=float),
    }


def finite_expert_cap(expert_radii, res_cal):
    finite_values = expert_radii[np.isfinite(expert_radii)]

    if len(finite_values) == 0:
        return 1e6

    return 10.0 * max(float(np.max(finite_values)), float(np.max(res_cal)), eps)


def run_agaci_stepwise(res_cal, res_test, alpha_target, gammas, eta, update_history=True):
    res_cal = np.asarray(res_cal, dtype=float)
    res_test = np.asarray(res_test, dtype=float)

    experts = run_aci_experts_stepwise(
        res_cal=res_cal,
        res_test=res_test,
        alpha_target=alpha_target,
        gammas=gammas,
        update_history=update_history,
    )

    expert_radii = experts["expert_radii"]
    K, n_test_local, T_local = expert_radii.shape

    cap = finite_expert_cap(expert_radii, res_cal)

    weights = np.ones((K, T_local), dtype=float) / K
    weight_history = np.zeros((n_test_local, K, T_local), dtype=float)
    radius = np.zeros((n_test_local, T_local), dtype=float)

    tau_upper = 1.0 - alpha_target / 2.0

    for i in range(n_test_local):
        for t in range(T_local):
            w = weights[:, t]
            w = w / w.sum()

            expert_pred = expert_radii[:, i, t]
            expert_pred = np.where(np.isfinite(expert_pred), expert_pred, cap)

            radius[i, t] = float(np.sum(w * expert_pred))
            weight_history[i, :, t] = w

            losses = pinball_loss(
                y=res_test[i, t],
                q=expert_pred,
                tau=tau_upper,
            )

            scale = max(float(np.mean(losses)), eps)

            weights[:, t] *= np.exp(-eta * losses / scale)

            if weights[:, t].sum() <= 0 or not np.isfinite(weights[:, t].sum()):
                weights[:, t] = np.ones(K, dtype=float) / K
            else:
                weights[:, t] /= weights[:, t].sum()

    return {
        "radius": radius,
        "alpha_history": experts["expert_alpha_history"],
        "weights": weights,
        "weight_history": weight_history,
        "experts": experts,
        "gammas": np.asarray(gammas, dtype=float),
    }


def run_agaci_trajectory(res_cal, res_test, alpha_target, gammas, eta, update_history=True):
    res_cal = np.asarray(res_cal, dtype=float)
    res_test = np.asarray(res_test, dtype=float)

    experts = run_aci_experts_trajectory(
        res_cal=res_cal,
        res_test=res_test,
        alpha_target=alpha_target,
        gammas=gammas,
        update_history=update_history,
    )

    expert_radii = experts["expert_radii"]
    K, n_test_local, T_local = expert_radii.shape

    cap = finite_expert_cap(expert_radii, res_cal)

    weights = np.ones(K, dtype=float) / K
    weight_history = np.zeros((n_test_local, K), dtype=float)
    score_radius = np.zeros(n_test_local, dtype=float)

    tau_upper = 1.0 - alpha_target / 2.0
    scores_test = np.max(res_test, axis=1)

    for i in range(n_test_local):
        w = weights / weights.sum()

        expert_pred = expert_radii[:, i, 0]
        expert_pred = np.where(np.isfinite(expert_pred), expert_pred, cap)

        score_radius[i] = float(np.sum(w * expert_pred))
        weight_history[i] = w

        losses = pinball_loss(
            y=scores_test[i],
            q=expert_pred,
            tau=tau_upper,
        )

        scale = max(float(np.mean(losses)), eps)

        weights *= np.exp(-eta * losses / scale)

        if weights.sum() <= 0 or not np.isfinite(weights.sum()):
            weights = np.ones(K, dtype=float) / K
        else:
            weights /= weights.sum()

    radius = np.tile(score_radius[:, None], (1, T_local))

    return {
        "radius": radius,
        "alpha_history": experts["expert_alpha_history"],
        "weights": weights,
        "weight_history": weight_history,
        "experts": experts,
        "gammas": np.asarray(gammas, dtype=float),
    }


def copulacpts_search_alpha_torch(alpha_input, epsilon, epochs=800, lr=1e-3):
    if torch is None or nn is None:
        raise ImportError("torch is required for torch-based CopulaCPTS search_alpha.")

    alpha_input = np.asarray(alpha_input, dtype=float)
    pseudo_data = torch.tensor(alpha_input, dtype=torch.float32)

    dim = alpha_input.shape[-1]
    alphas = nn.Parameter(torch.ones(dim, dtype=torch.float32))
    optimizer = torch.optim.Adam([alphas], lr=lr, weight_decay=1e-4)

    for _ in range(epochs):
        optimizer.zero_grad()

        coverage_smooth = torch.mean(
            torch.relu(
                torch.prod(
                    torch.sigmoid((alphas[None, :] - pseudo_data) * 1000.0),
                    dim=1,
                )
            )
        )

        loss = torch.abs(coverage_smooth - 1.0 + epsilon)
        loss.backward()
        optimizer.step()

    return alphas.detach().cpu().numpy()


def copulacpts_search_alpha_grid(alpha_input, epsilon, grid_size=401):
    alpha_input = np.asarray(alpha_input, dtype=float)
    grid = np.linspace(0.0, 1.0, grid_size)

    best_c = 1.0
    best_loss = np.inf

    for c in grid:
        coverage = np.mean(np.all(alpha_input <= c, axis=1))
        loss = abs(coverage - (1.0 - epsilon))

        if loss < best_loss:
            best_loss = loss
            best_c = c

    return np.full(alpha_input.shape[1], best_c, dtype=float)


def copulacpts_search_alpha(alpha_input, epsilon, epochs=800):
    if torch is not None and nn is not None:
        return copulacpts_search_alpha_torch(
            alpha_input=alpha_input,
            epsilon=epsilon,
            epochs=epochs,
        )

    return copulacpts_search_alpha_grid(
        alpha_input=alpha_input,
        epsilon=epsilon,
    )


def run_copulacpts_residual_norm(
    res_cal,
    res_test,
    alpha_level=0.10,
    split=0.60,
    epochs=800,
    random_state=123,
):
    res_cal = np.asarray(res_cal, dtype=float)
    res_test = np.asarray(res_test, dtype=float)

    n_cal_local, T_local = res_cal.shape
    n_test_local = res_test.shape[0]

    if n_cal_local < 3:
        raise ValueError("CopulaCPTS needs at least 3 calibration samples.")

    rng = np.random.default_rng(random_state)
    idx = rng.permutation(n_cal_local)

    n_quantile = int(split * n_cal_local)
    n_quantile = min(max(n_quantile, 1), n_cal_local - 1)

    idx_quantile = idx[:n_quantile]
    idx_copula = idx[n_quantile:]

    nonconformity = res_cal[idx_quantile]
    copula_scores = res_cal[idx_copula]

    alphas = np.zeros_like(copula_scores, dtype=float)

    for i in range(copula_scores.shape[0]):
        alphas[i] = (copula_scores[i] > nonconformity).mean(axis=0)

    threshold = copulacpts_search_alpha(
        alpha_input=alphas,
        epsilon=alpha_level,
        epochs=epochs,
    )

    threshold = np.asarray(threshold, dtype=float)
    threshold = np.clip(threshold, 0.0, 1.0)

    radius_h = np.zeros(T_local, dtype=float)
    mapping_shape = nonconformity.shape[0]

    for t in range(T_local):
        sorted_scores_t = np.sort(nonconformity[:, t])

        q_idx = int(threshold[t] * mapping_shape) + 1
        q_idx = min(max(q_idx, 0), mapping_shape - 1)

        radius_h[t] = sorted_scores_t[q_idx]

    radius = np.tile(radius_h[None, :], (n_test_local, 1))
    covered = res_test <= radius

    return {
        "radius": radius,
        "covered": covered,
        "radius_h": radius_h,
        "threshold": threshold,
        "alphas": alphas,
        "nonconformity": nonconformity,
        "copula_scores": copula_scores,
        "idx_quantile": idx_quantile,
        "idx_copula": idx_copula,
        "pointwise_coverage": covered.mean(),
        "sequence_coverage": np.all(covered, axis=1).mean(),
        "trajectory_failure": (~np.all(covered, axis=1)).mean(),
        "avg_radius": radius.mean(),
    }


def evaluate_radius_result(name, category, radius, residuals):
    radius = np.asarray(radius, dtype=float)
    residuals = np.asarray(residuals, dtype=float)
    if radius.shape != residuals.shape:
        raise ValueError(
            f"{name}: radius shape {radius.shape} != residual shape {residuals.shape}"
        )

    covered = residuals <= radius
    horizon_coverage = covered.mean(axis=0)
    horizon_radius = radius.mean(axis=0)
    trajectory_covered = np.all(covered, axis=1)

    return {
        "category": category,
        "method": name,
        "display_name": DISPLAY_NAME[name],
        "mhc": float(np.mean(horizon_coverage)),
        "mhc_horizon_std": float(np.std(horizon_coverage)),
        "whc": float(np.min(horizon_coverage)),
        "trajectory_coverage": float(np.mean(trajectory_covered)),
        "trajectory_failure": float(np.mean(~trajectory_covered)),
        "afr": float(np.mean(horizon_radius)),
        "afr_horizon_std": float(np.std(horizon_radius)),
        "maximum_horizon_afr": float(np.max(horizon_radius)),
        "per_horizon_coverage": horizon_coverage,
        "per_horizon_radius": horizon_radius,
        "lambda_star": None,
        "formal": None,
        "selection_mode": None,
        "ltt_empirical_risk": None,
        "ltt_p_value": None,
        "accepted_lambdas": [],
    }


In [ ]:
# =========================================================
# RANDOM TRAJECTORY SPLITS AND TRACE-CRC LTT HELPERS
# =========================================================

def make_trajectory_split(n_total, split_seed):
    if not 0 < CALIBRATION_SIZE < n_total:
        raise ValueError("Calibration size must be between zero and n_total.")

    rng = np.random.default_rng(int(split_seed))
    permutation = rng.permutation(n_total)
    calibration_indices = permutation[:CALIBRATION_SIZE]
    # Preserve a stable traversal order for methods with online updates.
    test_indices = np.sort(permutation[CALIBRATION_SIZE:])

    if np.intersect1d(calibration_indices, test_indices).size:
        raise RuntimeError("Calibration and test trajectories overlap.")
    if len(calibration_indices) + len(test_indices) != n_total:
        raise RuntimeError("Trajectory split does not cover the full dataset.")

    return calibration_indices, test_indices


def trajectory_features(predicted_norm):
    predicted_norm = np.asarray(predicted_norm, dtype=float)
    mean_value = predicted_norm.mean(axis=1)
    std_value = predicted_norm.std(axis=1)
    max_value = predicted_norm.max(axis=1)
    min_value = predicted_norm.min(axis=1)
    value_range = max_value - min_value
    slope = predicted_norm[:, -1] - predicted_norm[:, 0]
    mean_abs_difference = np.mean(
        np.abs(np.diff(predicted_norm, axis=1)),
        axis=1,
    )
    curvature = (
        np.mean(
            np.abs(np.diff(predicted_norm, n=2, axis=1)),
            axis=1,
        )
        if predicted_norm.shape[1] >= 3
        else np.zeros(predicted_norm.shape[0])
    )

    return np.column_stack(
        [
            mean_value,
            std_value,
            max_value,
            min_value,
            value_range,
            slope,
            mean_abs_difference,
            curvature,
            predicted_norm[:, 0],
            predicted_norm[:, -1],
        ]
    )


def conformal_quantile_plus(values, alpha_level):
    values = np.asarray(values, dtype=float).reshape(-1)
    if len(values) == 0:
        raise ValueError("Cannot calibrate an empty group.")
    order = math.ceil((len(values) + 1) * (1.0 - alpha_level))
    if order > len(values):
        return float("inf")
    return float(np.sort(values)[order - 1])


def moving_average(values, window):
    values = np.asarray(values, dtype=float)
    window = int(window)
    if window <= 1:
        return values.copy()
    if window % 2 == 0:
        window += 1
    half_width = window // 2
    return np.array(
        [
            np.mean(
                values[
                    max(0, index - half_width):
                    min(len(values), index + half_width + 1)
                ]
            )
            for index in range(len(values))
        ],
        dtype=float,
    )


def fit_trace_structure(cal_residual, cal_predicted_norm, eval_predicted_norm):
    cal_residual = np.asarray(cal_residual, dtype=float)
    cal_predicted_norm = np.asarray(cal_predicted_norm, dtype=float)
    eval_predicted_norm = np.asarray(eval_predicted_norm, dtype=float)

    if cal_residual.shape != cal_predicted_norm.shape:
        raise ValueError("TRACE calibration residual/predicted-norm shapes differ.")
    if eval_predicted_norm.shape[1] != cal_residual.shape[1]:
        raise ValueError("TRACE evaluation horizon differs from calibration.")
    if trace_num_groups != 2:
        raise ValueError("The paper-aligned TRACE implementation uses two groups.")

    n_cal_local, horizon_count = cal_residual.shape
    n_prof = int(round(trace_profile_fraction * n_cal_local))
    n_remaining = n_cal_local - n_prof
    n_cp = int(round(trace_cp_fraction_within_rest * n_remaining))
    n_ltt = n_remaining - n_cp
    if min(n_prof, n_cp, n_ltt) <= 0:
        raise ValueError("Invalid TRACE profile/CP/LTT split.")

    prof_slice = slice(0, n_prof)
    cp_slice = slice(n_prof, n_prof + n_cp)
    ltt_slice = slice(n_prof + n_cp, n_cal_local)

    res_prof = cal_residual[prof_slice]
    res_cp = cal_residual[cp_slice]
    res_ltt = cal_residual[ltt_slice]
    pred_prof = cal_predicted_norm[prof_slice]
    pred_cp = cal_predicted_norm[cp_slice]
    pred_ltt = cal_predicted_norm[ltt_slice]

    w_profile = np.quantile(res_prof, 1.0 - alpha, axis=0)
    w_profile = moving_average(np.maximum(w_profile, eps), trace_smooth_window)
    w_profile = np.maximum(w_profile, trace_floor_ratio * np.median(w_profile))
    w_profile = w_profile / np.mean(w_profile)

    normalized_profile_target = np.max(
        res_prof / (w_profile[None, :] + eps),
        axis=1,
    )
    raw_profile_target = np.max(res_prof, axis=1)
    scaler = StandardScaler()
    profile_features = scaler.fit_transform(trajectory_features(pred_prof))
    difficulty_model = Ridge(alpha=trace_ridge_alpha)
    difficulty_model.fit(profile_features, normalized_profile_target)
    raw_difficulty_model = Ridge(alpha=trace_ridge_alpha)
    raw_difficulty_model.fit(profile_features, raw_profile_target)

    cp_difficulty = difficulty_model.predict(
        scaler.transform(trajectory_features(pred_cp))
    )
    ltt_difficulty = difficulty_model.predict(
        scaler.transform(trajectory_features(pred_ltt))
    )
    eval_difficulty = difficulty_model.predict(
        scaler.transform(trajectory_features(eval_predicted_norm))
    )
    raw_cp_difficulty = raw_difficulty_model.predict(
        scaler.transform(trajectory_features(pred_cp))
    )
    raw_ltt_difficulty = raw_difficulty_model.predict(
        scaler.transform(trajectory_features(pred_ltt))
    )
    raw_eval_difficulty = raw_difficulty_model.predict(
        scaler.transform(trajectory_features(eval_predicted_norm))
    )

    group_threshold = float(np.median(cp_difficulty))
    group_cp = (cp_difficulty > group_threshold).astype(int)
    group_ltt = (ltt_difficulty > group_threshold).astype(int)
    group_eval = (eval_difficulty > group_threshold).astype(int)
    raw_group_threshold = float(np.median(raw_cp_difficulty))
    raw_group_cp = (raw_cp_difficulty > raw_group_threshold).astype(int)
    raw_group_ltt = (raw_ltt_difficulty > raw_group_threshold).astype(int)
    raw_group_eval = (raw_eval_difficulty > raw_group_threshold).astype(int)

    cp_scores = np.max(
        res_cp / (w_profile[None, :] + eps),
        axis=1,
    )
    q_group = {
        group: conformal_quantile_plus(
            cp_scores[group_cp == group],
            alpha,
        )
        for group in range(trace_num_groups)
    }
    if not all(np.isfinite(list(q_group.values()))):
        raise RuntimeError("At least one TRACE group quantile is infinite.")

    raw_cp_scores = np.max(res_cp, axis=1)
    q_group_raw = {
        group: conformal_quantile_plus(
            raw_cp_scores[raw_group_cp == group],
            alpha,
        )
        for group in range(trace_num_groups)
    }
    if not all(np.isfinite(list(q_group_raw.values()))):
        raise RuntimeError(
            "At least one groups-only quantile is infinite."
        )

    return {
        "n_prof": n_prof,
        "n_cp": n_cp,
        "n_ltt": n_ltt,
        "res_prof": res_prof,
        "res_cp": res_cp,
        "res_ltt": res_ltt,
        "res_cal": cal_residual,
        "w_profile": w_profile,
        "q_group": q_group,
        "q_group_raw": q_group_raw,
        "group_cp": group_cp,
        "group_ltt": group_ltt,
        "group_eval": group_eval,
        "raw_group_cp": raw_group_cp,
        "raw_group_ltt": raw_group_ltt,
        "raw_group_eval": raw_group_eval,
        "q_global": conformal_order_stat(res_cp.reshape(-1), alpha),
        "horizon_count": horizon_count,
    }


def trace_style_radius(style, lam, components, groups, n_rows):
    horizon_count = components["horizon_count"]
    if style == "global":
        return np.full(
            (n_rows, horizon_count),
            lam * components["q_global"],
            dtype=float,
        )
    if style == "horizon":
        horizon_radius = (
            lam * components["q_global"] * components["w_profile"]
        )
        return np.tile(horizon_radius[None, :], (n_rows, 1))
    if style == "groups":
        return np.vstack(
            [
                np.full(
                    horizon_count,
                    lam * components["q_group_raw"][int(group)],
                )
                for group in groups
            ]
        )
    if style == "full":
        return np.vstack(
            [
                (
                    lam
                    * components["q_group"][int(group)]
                    * components["w_profile"]
                )
                for group in groups
            ]
        )
    raise ValueError(f"Unknown TRACE/LTT style: {style}")


def select_trace_ltt(style, components):
    if style == "global" and use_all_calib_global_crc:
        res_ltt = components["res_cal"]
        ltt_data_source = "all_calibration"
    else:
        res_ltt = components["res_ltt"]
        ltt_data_source = "validation_partition"

    n_ltt = len(res_ltt)
    if style == "groups":
        groups = components["raw_group_ltt"]
    elif style == "full":
        groups = components["group_ltt"]
    else:
        groups = None

    active_lambda_grid = np.asarray(lambda_grid_master, dtype=float)
    if np.any(np.diff(active_lambda_grid) <= 0):
        raise ValueError("The prespecified LTT master grid must be increasing.")

    loss_table = np.zeros(
        (n_ltt, len(active_lambda_grid)),
        dtype=float,
    )
    for lambda_index, lambda_value in enumerate(active_lambda_grid):
        radius = trace_style_radius(
            style,
            lambda_value,
            components,
            groups,
            n_ltt,
        )
        loss_table[:, lambda_index] = np.any(
            res_ltt > radius,
            axis=1,
        ).astype(float)

    empirical_risks = loss_table.mean(axis=0)
    with np.errstate(divide="ignore", invalid="ignore"):
        p_values = np.array(
            [
                hb_p_value(float(risk), n_ltt, alpha)
                for risk in empirical_risks
            ],
            dtype=float,
        )
        accepted_raw = romano_wolf_HB(
            loss_table,
            active_lambda_grid,
            alpha=alpha,
            delta=delta_ltt,
        )
    accepted_indices = sorted(
        {
            int(index)
            for index in accepted_raw
            if 0 <= int(index) < len(active_lambda_grid)
        }
    )

    if not accepted_indices:
        raise RuntimeError(
            f"{style} CRC failed to find a formally accepted lambda in "
            f"the prespecified master grid through {lambda_grid_max_cap:.2f}. "
            "No empirical fallback is permitted."
        )

    selected_index = accepted_indices[0]
    selected_lambda = float(active_lambda_grid[selected_index])
    lambda_expansions = max(
        0,
        int(
            round(
                (selected_lambda - float(lambda_grid_initial[-1]))
                / lambda_grid_expand_step
            )
        ),
    )
    formal = True
    selection_mode = (
        "formal-expanded-grid"
        if lambda_expansions > 0
        else "formal-initial-grid"
    )

    return {
        "lambda_star": selected_lambda,
        "formal": bool(formal),
        "selection_mode": selection_mode,
        "ltt_empirical_risk": float(empirical_risks[selected_index]),
        "ltt_p_value": float(p_values[selected_index]),
        "accepted_lambdas": [
            float(active_lambda_grid[index])
            for index in accepted_indices
        ],
        "lambda_grid_initial": lambda_grid_initial.copy(),
        "lambda_grid_extension": lambda_grid_extension.copy(),
        "lambda_grid_master": active_lambda_grid.copy(),
        "lambda_grid_used": active_lambda_grid.copy(),
        "lambda_family_size": int(len(active_lambda_grid)),
        "lambda_initial_max": float(lambda_grid_initial[-1]),
        "lambda_max_cap": float(lambda_grid_max_cap),
        "lambda_expand_step": float(lambda_grid_expand_step),
        "lambda_grid_expansions": int(lambda_expansions),
        "expansion_policy": "prespecified_master_grid_joint_holm_hb",
        "multiple_testing_method": "holm_stepdown_hb",
        "empirical_fallback_used": False,
        "ltt_data_source": ltt_data_source,
        "n_ltt_used": int(n_ltt),
        "loss_table": loss_table,
    }


def run_trace_style(
    name,
    style,
    cal_residual,
    cal_predicted_norm,
    eval_residual,
    eval_predicted_norm,
):
    components = fit_trace_structure(
        cal_residual,
        cal_predicted_norm,
        eval_predicted_norm,
    )
    selection = select_trace_ltt(style, components)
    if style == "groups":
        eval_groups = components["raw_group_eval"]
    elif style == "full":
        eval_groups = components["group_eval"]
    else:
        eval_groups = None
    radius = trace_style_radius(
        style,
        selection["lambda_star"],
        components,
        eval_groups,
        len(eval_residual),
    )
    result = evaluate_radius_result(
        name,
        C_RISK,
        radius,
        eval_residual,
    )
    active_q_group = (
        components["q_group_raw"]
        if style == "groups"
        else components["q_group"] if style == "full" else None
    )
    result.update(
        {
            key: selection[key]
            for key in (
                "lambda_star",
                "formal",
                "selection_mode",
                "ltt_empirical_risk",
                "ltt_p_value",
                "accepted_lambdas",
                "lambda_grid_initial",
                "lambda_grid_extension",
                "lambda_grid_master",
                "lambda_grid_used",
                "lambda_family_size",
                "lambda_initial_max",
                "lambda_max_cap",
                "lambda_expand_step",
                "lambda_grid_expansions",
                "expansion_policy",
                "multiple_testing_method",
                "empirical_fallback_used",
                "ltt_data_source",
            )
        }
    )
    result.update(
        {
            "n_prof": components["n_prof"],
            "n_cp": components["n_cp"],
            "n_ltt": selection["n_ltt_used"],
            "q_global": float(components["q_global"]),
            "q_group_0": (
                None if active_q_group is None else float(active_q_group[0])
            ),
            "q_group_1": (
                None if active_q_group is None else float(active_q_group[1])
            ),
        }
    )
    return result


In [ ]:
# =========================================================
# FIVE-SEED CALIBRATION-ONLY TUNING FOR PAPER BASELINES
# =========================================================

RUN_BASELINE_PARAMETER_SEARCH = USE_FIVE_SEED_BASELINE_TUNING
BASELINE_TUNING_SEEDS = tuple(INFERENCE_SEEDS)
BASELINE_TUNING_VALIDATION_FRACTION = 0.30
BASELINE_TUNING_MIN_TRAIN = 50
BASELINE_TUNING_MIN_VALIDATION = 20
BASELINE_TUNING_COVERAGE_TOLERANCE = 0.0

BASELINE_WEIGHTED_BETA_GRID = [0.003, 0.005, 0.01, 0.02, 0.03, 0.05]
BASELINE_ENBPI_B_GRID = [50, 100]
BASELINE_ENBPI_WINDOW_GRID = [None, 100, 200, 300]
BASELINE_ENBPI_CENTER_GRID = ["mean", "oob_quantile"]
BASELINE_ACI_GAMMA_GRID = [0.001, 0.003, 0.005, 0.01, 0.02, 0.05]
BASELINE_AGACI_GAMMA_SETS = [
    np.array([0.0, 0.0005, 0.001, 0.002, 0.005, 0.01]),
    np.array([0.0, 0.001, 0.003, 0.005, 0.01, 0.02, 0.03]),
    np.array([0.0, 0.005, 0.01, 0.02, 0.03, 0.05]),
]
BASELINE_AGACI_ETA_GRID = [0.05, 0.10, 0.20, 0.50]
BASELINE_COPULACPTS_SPLIT_GRID = [0.50, 0.60, 0.70, 0.80]
BASELINE_COPULACPTS_EPOCH_GRID = [400, 800, 1200]


def baseline_tuning_split(residuals):
    residuals = np.asarray(residuals, dtype=float)
    n_rows = len(residuals)
    if n_rows < BASELINE_TUNING_MIN_TRAIN + BASELINE_TUNING_MIN_VALIDATION:
        raise ValueError("Insufficient calibration trajectories for baseline tuning.")
    n_validation = int(
        np.ceil(BASELINE_TUNING_VALIDATION_FRACTION * n_rows)
    )
    n_validation = min(
        max(n_validation, BASELINE_TUNING_MIN_VALIDATION),
        n_rows - BASELINE_TUNING_MIN_TRAIN,
    )
    return residuals[:-n_validation], residuals[-n_validation:]


def evaluate_baseline_tuning_radius(residuals, radius):
    residuals = np.asarray(residuals, dtype=float)
    radius = np.asarray(radius, dtype=float)
    if residuals.shape != radius.shape:
        raise ValueError(
            f"Tuning residual/radius shape mismatch: {residuals.shape} vs {radius.shape}."
        )
    covered = residuals <= radius
    trajectory_covered = np.all(covered, axis=1)
    horizon_coverage = np.mean(covered, axis=0)
    return {
        "pointwise_coverage": float(np.mean(covered)),
        "worst_horizon_coverage": float(np.min(horizon_coverage)),
        "trajectory_coverage": float(np.mean(trajectory_covered)),
        "trajectory_failure": float(np.mean(~trajectory_covered)),
        "average_radius": float(np.mean(radius)),
    }


def summarize_baseline_candidate(seed_metrics, objective):
    if objective == "pointwise":
        target_metric = "pointwise_coverage"
    elif objective == "trajectory":
        target_metric = "trajectory_coverage"
    else:
        raise ValueError(f"Unknown tuning objective: {objective}.")

    target_coverages = np.asarray(
        [row[target_metric] for row in seed_metrics],
        dtype=float,
    )
    radii = np.asarray(
        [row["average_radius"] for row in seed_metrics],
        dtype=float,
    )
    worst_horizon_coverages = np.asarray(
        [row["worst_horizon_coverage"] for row in seed_metrics],
        dtype=float,
    )
    shortfalls = np.maximum(
        0.0,
        (1.0 - alpha) - target_coverages,
    )
    all_seed_feasible = bool(
        np.all(shortfalls <= BASELINE_TUNING_COVERAGE_TOLERANCE)
    )
    summary = {
        "objective": objective,
        "target_metric": target_metric,
        "all_seed_feasible": all_seed_feasible,
        "minimum_target_coverage": float(np.min(target_coverages)),
        "mean_target_coverage": float(np.mean(target_coverages)),
        "std_target_coverage": float(np.std(target_coverages)),
        "maximum_seed_shortfall": float(np.max(shortfalls)),
        "mean_seed_shortfall": float(np.mean(shortfalls)),
        "mean_average_radius": float(np.mean(radii)),
        "std_average_radius": float(np.std(radii)),
        "minimum_worst_horizon_coverage": float(
            np.min(worst_horizon_coverages)
        ),
    }
    if all_seed_feasible:
        key = (
            0,
            summary["mean_average_radius"],
            -summary["minimum_target_coverage"],
            summary["std_average_radius"],
        )
    else:
        key = (
            1,
            summary["maximum_seed_shortfall"],
            summary["mean_seed_shortfall"],
            summary["mean_average_radius"],
        )
    return summary, key


def tune_baseline_over_grid(parameter_grid, runner, objective):
    candidates = []
    for parameters in parameter_grid:
        seed_metrics = []
        for inference_seed in BASELINE_TUNING_SEEDS:
            tuning_data = baseline_tuning_data[inference_seed]
            radius = runner(
                parameters,
                inference_seed,
                tuning_data["train"],
                tuning_data["validation"],
            )
            metrics = evaluate_baseline_tuning_radius(
                tuning_data["validation"],
                radius,
            )
            seed_metrics.append(
                {
                    "inference_seed": int(inference_seed),
                    **metrics,
                }
            )
        summary, key = summarize_baseline_candidate(
            seed_metrics,
            objective,
        )
        candidates.append(
            {
                "params": copy.deepcopy(parameters),
                "seed_metrics": seed_metrics,
                "summary": summary,
                "key": key,
            }
        )
    if not candidates:
        raise RuntimeError("A baseline tuning grid was empty.")
    return min(candidates, key=lambda row: row["key"]), candidates


if not RUN_BASELINE_PARAMETER_SEARCH:
    raise RuntimeError(
        "Five-seed baseline tuning must be enabled before test evaluation."
    )

baseline_cal_indices, _ = make_trajectory_split(
    reference_shape[0],
    PRIMARY_SPLIT_SEED,
)
baseline_tuning_data = {}
for inference_seed in BASELINE_TUNING_SEEDS:
    calibration_residuals = np.asarray(
        seed_compact_data[inference_seed]["fro_residual"],
        dtype=float,
    )[baseline_cal_indices]
    tuning_train, tuning_validation = baseline_tuning_split(
        calibration_residuals
    )
    baseline_tuning_data[inference_seed] = {
        "train": tuning_train,
        "validation": tuning_validation,
    }

baseline_tune_train_size = len(
    baseline_tuning_data[BASELINE_TUNING_SEEDS[0]]["train"]
)
baseline_tune_validation_size = len(
    baseline_tuning_data[BASELINE_TUNING_SEEDS[0]]["validation"]
)
if any(
    len(value["train"]) != baseline_tune_train_size
    or len(value["validation"]) != baseline_tune_validation_size
    for value in baseline_tuning_data.values()
):
    raise RuntimeError("Baseline tuning split sizes differ across seeds.")

baseline_tuning_results = {}
baseline_tuning_candidates = {}


def store_baseline_search(name, grid, runner, objective):
    best, candidates = tune_baseline_over_grid(
        grid,
        runner,
        objective,
    )
    baseline_tuning_results[name] = best
    baseline_tuning_candidates[name] = candidates
    return best


weighted_horizon_best = store_baseline_search(
    "weighted_horizon",
    [{"beta": beta} for beta in BASELINE_WEIGHTED_BETA_GRID],
    lambda p, seed, train, validation: run_weighted_cp(
        train,
        validation,
        alpha,
        p["beta"],
        mode="stepwise",
    ),
    "pointwise",
)
weighted_trajectory_best = store_baseline_search(
    "weighted_trajectory",
    [{"beta": beta} for beta in BASELINE_WEIGHTED_BETA_GRID],
    lambda p, seed, train, validation: run_weighted_cp(
        train,
        validation,
        alpha,
        p["beta"],
        mode="trajectory",
    ),
    "trajectory",
)

enbpi_grid = [
    {"B": B, "window": window, "center_method": center}
    for B in BASELINE_ENBPI_B_GRID
    for window in BASELINE_ENBPI_WINDOW_GRID
    for center in BASELINE_ENBPI_CENTER_GRID
]
enbpi_horizon_best = store_baseline_search(
    "enbpi_horizon",
    enbpi_grid,
    lambda p, seed, train, validation: run_enbpi_csi_stepwise(
        fro_res_cal=train,
        fro_res_test=validation,
        base_model=Ridge(alpha=1.0),
        alpha_level=alpha,
        B=p["B"],
        window=p["window"],
        online_update=True,
        center_method=p["center_method"],
        random_state=123,
    )["radius"],
    "pointwise",
)
enbpi_trajectory_best = store_baseline_search(
    "enbpi_trajectory",
    enbpi_grid,
    lambda p, seed, train, validation: run_enbpi_csi_trajectory(
        fro_res_cal=train,
        fro_res_test=validation,
        base_model=Ridge(alpha=1.0),
        alpha_level=alpha,
        B=p["B"],
        window=p["window"],
        online_update=True,
        center_method=p["center_method"],
        random_state=456,
    )["radius"],
    "trajectory",
)

aci_horizon_best = store_baseline_search(
    "aci_horizon",
    [{"gamma": gamma} for gamma in BASELINE_ACI_GAMMA_GRID],
    lambda p, seed, train, validation: run_aci_stepwise(
        train,
        validation,
        alpha,
        p["gamma"],
        update_history=True,
    )["radius"],
    "pointwise",
)
aci_trajectory_best = store_baseline_search(
    "aci_trajectory",
    [{"gamma": gamma} for gamma in BASELINE_ACI_GAMMA_GRID],
    lambda p, seed, train, validation: run_aci_trajectory(
        train,
        validation,
        alpha,
        p["gamma"],
        update_history=True,
    )["radius"],
    "trajectory",
)

agaci_grid = [
    {"gammas": gammas, "eta": eta}
    for gammas in BASELINE_AGACI_GAMMA_SETS
    for eta in BASELINE_AGACI_ETA_GRID
]
agaci_horizon_best = store_baseline_search(
    "agaci_horizon",
    agaci_grid,
    lambda p, seed, train, validation: run_agaci_stepwise(
        train,
        validation,
        alpha,
        p["gammas"],
        p["eta"],
        update_history=True,
    )["radius"],
    "pointwise",
)
agaci_trajectory_best = store_baseline_search(
    "agaci_trajectory",
    agaci_grid,
    lambda p, seed, train, validation: run_agaci_trajectory(
        train,
        validation,
        alpha,
        p["gammas"],
        p["eta"],
        update_history=True,
    )["radius"],
    "trajectory",
)

copulacpts_best = store_baseline_search(
    "copulacpts",
    [
        {"split": split, "epochs": epochs}
        for split in BASELINE_COPULACPTS_SPLIT_GRID
        for epochs in BASELINE_COPULACPTS_EPOCH_GRID
    ],
    lambda p, seed, train, validation: run_copulacpts_residual_norm(
        train,
        validation,
        alpha,
        split=p["split"],
        epochs=p["epochs"],
        random_state=123,
    )["radius"],
    "trajectory",
)

BASELINE_TUNING_SELECTED = {
    "weighted_horizon_beta": weighted_horizon_best["params"]["beta"],
    "weighted_trajectory_beta": weighted_trajectory_best["params"]["beta"],
    "enbpi_horizon": enbpi_horizon_best["params"],
    "enbpi_trajectory": enbpi_trajectory_best["params"],
    "aci_horizon_gamma": aci_horizon_best["params"]["gamma"],
    "aci_trajectory_gamma": aci_trajectory_best["params"]["gamma"],
    "agaci_horizon": agaci_horizon_best["params"],
    "agaci_trajectory": agaci_trajectory_best["params"],
    "copulacpts": copulacpts_best["params"],
}

print("\nFIVE-SEED BASELINE PARAMETER TUNING")
print("Inference seeds:", list(BASELINE_TUNING_SEEDS))
print("Outer calibration split seed:", PRIMARY_SPLIT_SEED)
print(
    "Per-seed tuning train/validation sizes:",
    baseline_tune_train_size,
    baseline_tune_validation_size,
)
for method_name, result in baseline_tuning_results.items():
    summary = result["summary"]
    print(
        f"{method_name:<24} "
        f"params={result['params']} | "
        f"all-seed feasible={summary['all_seed_feasible']} | "
        f"min coverage={summary['minimum_target_coverage']:.4f} | "
        f"mean AFR={summary['mean_average_radius']:.4f}"
    )


In [ ]:
# =========================================================
# TRACE-CRC CERTIFIED PARAMETER SEARCH
# Run after the compact-cache and TRACE-helper cells.
# This cell never reads test residuals. Its selected structure is frozen in the next cell.
# =========================================================

from itertools import product

RUN_TRACE_PARAMETER_SEARCH = True

if RUN_TRACE_PARAMETER_SEARCH:
    SEARCH_ALPHA = 0.10
    SEARCH_DELTA = 0.10
    SEARCH_MAX_HORIZON_AFR = 25.0

    # Pre-specify this family before inspecting the search output.
    SEARCH_PROFILE_FRACTIONS = [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40]
    SEARCH_CP_FRACTIONS = [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40]
    SEARCH_SMOOTH_WINDOWS = [3]
    SEARCH_FLOOR_RATIOS = [0.15, 0.25]
    SEARCH_RIDGE_ALPHAS = [0.1, 1.0]
    SEARCH_LAMBDA_GRID = np.linspace(0.70, 1.30, 10)

    # Groups remain fixed because changing this changes the paper method.
    SEARCH_NUM_GROUPS = 2
    if SEARCH_NUM_GROUPS != 2:
        raise ValueError("The paper-aligned search requires two groups.")

    search_n_total, search_horizon_count = reference_shape
    search_cal_indices, _ = make_trajectory_split(
        search_n_total,
        PRIMARY_SPLIT_SEED,
    )
    if len(search_cal_indices) != CALIBRATION_SIZE:
        raise RuntimeError("Search calibration split has the wrong size.")

    search_configs = list(
        product(
            SEARCH_PROFILE_FRACTIONS,
            SEARCH_CP_FRACTIONS,
            SEARCH_SMOOTH_WINDOWS,
            SEARCH_FLOOR_RATIOS,
            SEARCH_RIDGE_ALPHAS,
        )
    )
    search_hypotheses = []
    search_skipped = []

    for config_id, config_values in enumerate(search_configs):
        (
            profile_fraction,
            cp_fraction,
            smooth_window,
            floor_ratio,
            ridge_alpha,
        ) = config_values

        n_prof_search = int(
            round(profile_fraction * CALIBRATION_SIZE)
        )
        n_remaining_search = CALIBRATION_SIZE - n_prof_search
        n_cp_search = int(round(cp_fraction * n_remaining_search))
        n_ltt_search = n_remaining_search - n_cp_search

        if min(n_prof_search, n_cp_search, n_ltt_search) < 10:
            search_skipped.append(
                {
                    "config_id": config_id,
                    "reason": "a split contains fewer than 10 trajectories",
                }
            )
            continue

        lambda_seed_p_values = [
            [] for _ in SEARCH_LAMBDA_GRID
        ]
        lambda_seed_risks = [
            [] for _ in SEARCH_LAMBDA_GRID
        ]
        lambda_seed_afrs = [
            [] for _ in SEARCH_LAMBDA_GRID
        ]
        lambda_seed_horizon_afrs = [
            [] for _ in SEARCH_LAMBDA_GRID
        ]

        try:
            for inference_seed in INFERENCE_SEEDS:
                residual_all = np.asarray(
                    seed_compact_data[inference_seed]["fro_residual"],
                    dtype=float,
                )
                predicted_norm_all = np.asarray(
                    seed_compact_data[inference_seed]["predicted_norm"],
                    dtype=float,
                )
                residual_cal = residual_all[search_cal_indices]
                predicted_norm_cal = predicted_norm_all[
                    search_cal_indices
                ]

                prof_slice = slice(0, n_prof_search)
                cp_slice = slice(
                    n_prof_search,
                    n_prof_search + n_cp_search,
                )
                ltt_slice = slice(
                    n_prof_search + n_cp_search,
                    CALIBRATION_SIZE,
                )

                residual_prof = residual_cal[prof_slice]
                residual_cp = residual_cal[cp_slice]
                residual_ltt = residual_cal[ltt_slice]
                predicted_prof = predicted_norm_cal[prof_slice]
                predicted_cp = predicted_norm_cal[cp_slice]
                predicted_ltt = predicted_norm_cal[ltt_slice]

                w_search = np.quantile(
                    residual_prof,
                    1.0 - SEARCH_ALPHA,
                    axis=0,
                )
                w_search = moving_average(
                    np.maximum(w_search, eps),
                    smooth_window,
                )
                w_search = np.maximum(
                    w_search,
                    floor_ratio * np.median(w_search),
                )
                w_search = w_search / np.mean(w_search)

                target_prof = np.max(
                    residual_prof / (w_search[None, :] + eps),
                    axis=1,
                )
                scaler_search = StandardScaler()
                features_prof = scaler_search.fit_transform(
                    trajectory_features(predicted_prof)
                )
                model_search = Ridge(alpha=ridge_alpha)
                model_search.fit(features_prof, target_prof)

                difficulty_cp = model_search.predict(
                    scaler_search.transform(
                        trajectory_features(predicted_cp)
                    )
                )
                difficulty_ltt = model_search.predict(
                    scaler_search.transform(
                        trajectory_features(predicted_ltt)
                    )
                )
                group_threshold = float(np.median(difficulty_cp))
                group_cp = (
                    difficulty_cp > group_threshold
                ).astype(int)
                group_ltt = (
                    difficulty_ltt > group_threshold
                ).astype(int)

                cp_scores = np.max(
                    residual_cp / (w_search[None, :] + eps),
                    axis=1,
                )
                q_search = {
                    group: conformal_quantile_plus(
                        cp_scores[group_cp == group],
                        SEARCH_ALPHA,
                    )
                    for group in range(SEARCH_NUM_GROUPS)
                }
                if not all(np.isfinite(list(q_search.values()))):
                    raise ValueError("a group quantile is infinite")

                base_radius = np.vstack(
                    [
                        q_search[int(group)] * w_search
                        for group in group_ltt
                    ]
                )

                for lambda_index, lambda_value in enumerate(
                    SEARCH_LAMBDA_GRID
                ):
                    radius = float(lambda_value) * base_radius
                    losses = np.any(
                        residual_ltt > radius,
                        axis=1,
                    ).astype(float)
                    empirical_risk = float(np.mean(losses))
                    p_value = float(
                        hb_p_value(
                            empirical_risk,
                            n_ltt_search,
                            SEARCH_ALPHA,
                        )
                    )
                    lambda_seed_p_values[lambda_index].append(
                        p_value
                    )
                    lambda_seed_risks[lambda_index].append(
                        empirical_risk
                    )
                    lambda_seed_afrs[lambda_index].append(
                        float(np.mean(radius))
                    )
                    lambda_seed_horizon_afrs[lambda_index].append(
                        np.mean(radius, axis=0)
                    )

            for lambda_index, lambda_value in enumerate(
                SEARCH_LAMBDA_GRID
            ):
                # Intersection-union p-value: certification requires
                # every fixed inference seed to pass.
                joint_p_value = float(
                    np.max(lambda_seed_p_values[lambda_index])
                )
                mean_horizon_afr = np.mean(
                    np.stack(
                        lambda_seed_horizon_afrs[lambda_index],
                        axis=0,
                    ),
                    axis=0,
                )
                search_hypotheses.append(
                    {
                        "config_id": config_id,
                        "lambda_index": lambda_index,
                        "lambda_value": float(lambda_value),
                        "joint_p_value": joint_p_value,
                        "seed_p_values": list(
                            lambda_seed_p_values[lambda_index]
                        ),
                        "worst_seed_empirical_risk": float(
                            np.max(
                                lambda_seed_risks[lambda_index]
                            )
                        ),
                        "mean_afr": float(
                            np.mean(
                                lambda_seed_afrs[lambda_index]
                            )
                        ),
                        "max_horizon_afr": float(
                            np.max(mean_horizon_afr)
                        ),
                        "profile_fraction": profile_fraction,
                        "cp_fraction_within_rest": cp_fraction,
                        "smooth_window": smooth_window,
                        "floor_ratio": floor_ratio,
                        "ridge_alpha": ridge_alpha,
                        "n_prof": n_prof_search,
                        "n_cp": n_cp_search,
                        "n_ltt": n_ltt_search,
                    }
                )
        except Exception as error:
            search_skipped.append(
                {
                    "config_id": config_id,
                    "reason": str(error),
                }
            )

    if not search_hypotheses:
        raise RuntimeError("No valid TRACE search hypotheses.")

    # Holm controls the full configuration/lambda search under arbitrary
    # dependence between hypotheses.
    hypothesis_count = len(search_hypotheses)
    p_order = np.argsort(
        [row["joint_p_value"] for row in search_hypotheses]
    )
    certified_indices = []
    for rank, hypothesis_index in enumerate(p_order):
        holm_threshold = SEARCH_DELTA / (
            hypothesis_count - rank
        )
        row = search_hypotheses[int(hypothesis_index)]
        if row["joint_p_value"] <= holm_threshold:
            row["holm_threshold"] = float(holm_threshold)
            certified_indices.append(int(hypothesis_index))
        else:
            break

    certified_rows = [
        search_hypotheses[index]
        for index in certified_indices
    ]
    if not certified_rows:
        raise RuntimeError(
            "No configuration/lambda pair was certified after global "
            "Holm correction. Reduce the pre-specified search family or "
            "increase independent calibration data; do not select from "
            "uncertified rows."
        )

    bounded_rows = [
        row
        for row in certified_rows
        if row["max_horizon_afr"] <= SEARCH_MAX_HORIZON_AFR
    ]
    selection_pool = bounded_rows if bounded_rows else certified_rows
    trace_search_best = min(
        selection_pool,
        key=lambda row: (
            row["mean_afr"],
            row["max_horizon_afr"],
            row["worst_seed_empirical_risk"],
            -row["n_ltt"],
        ),
    )

    selected_structure_rows = [
        row
        for row in certified_rows
        if (
            row["profile_fraction"]
            == trace_search_best["profile_fraction"]
            and row["cp_fraction_within_rest"]
            == trace_search_best["cp_fraction_within_rest"]
            and row["smooth_window"]
            == trace_search_best["smooth_window"]
            and row["floor_ratio"]
            == trace_search_best["floor_ratio"]
            and row["ridge_alpha"]
            == trace_search_best["ridge_alpha"]
        )
    ]
    certified_lambda_grid = np.array(
        sorted(
            {
                row["lambda_value"]
                for row in selected_structure_rows
                if (
                    row["lambda_value"]
                    >= trace_search_best["lambda_value"]
                )
            }
        ),
        dtype=float,
    )
    if not len(certified_lambda_grid):
        certified_lambda_grid = np.array(
            [trace_search_best["lambda_value"]],
            dtype=float,
        )

    print("========================================")
    print("CERTIFIED TRACE-CRC PARAMETER SEARCH")
    print("========================================")
    print(f"Configurations searched : {len(search_configs)}")
    print(f"Total hypotheses        : {hypothesis_count}")
    print(f"Certified hypotheses    : {len(certified_rows)}")
    print(f"Configurations skipped  : {len(search_skipped)}")
    print(f"All inference seeds pass: True")
    print(
        "Horizon limit satisfied : "
        f"{bool(bounded_rows)}"
    )
    print()
    print(
        "Selected split          : "
        f"{trace_search_best['n_prof']}/"
        f"{trace_search_best['n_cp']}/"
        f"{trace_search_best['n_ltt']}"
    )
    print(
        "Selected lambda         : "
        f"{trace_search_best['lambda_value']:.8f}"
    )
    print(
        "Worst-seed LTT risk     : "
        f"{trace_search_best['worst_seed_empirical_risk']:.6f}"
    )
    print(
        "Joint HB p-value        : "
        f"{trace_search_best['joint_p_value']:.8g}"
    )
    print(
        "Mean calibration AFR    : "
        f"{trace_search_best['mean_afr']:.6f}"
    )
    print(
        "Maximum horizon AFR     : "
        f"{trace_search_best['max_horizon_afr']:.6f}"
    )

    print("\nPARAMETERS TO COPY")
    print(
        "trace_profile_fraction = "
        f"{trace_search_best['profile_fraction']}"
    )
    print(
        "trace_cp_fraction_within_rest = "
        f"{trace_search_best['cp_fraction_within_rest']}"
    )
    print("trace_num_groups = 2")
    print(
        "trace_smooth_window = "
        f"{trace_search_best['smooth_window']}"
    )
    print(
        "trace_floor_ratio = "
        f"{trace_search_best['floor_ratio']}"
    )
    print(
        "trace_ridge_alpha = "
        f"{trace_search_best['ridge_alpha']}"
    )
    print(
        "lambda_grid = np.array("
        f"{certified_lambda_grid.tolist()}, dtype=float)"
    )
    print(
        "# Globally corrected selected lambda = "
        f"{trace_search_best['lambda_value']:.8f}"
    )
else:
    print(
        "TRACE parameter search is disabled. Set "
        "RUN_TRACE_PARAMETER_SEARCH = True and run this cell manually."
    )


In [ ]:
# =========================================================
# FREEZE FIVE-SEED-SELECTED PARAMETERS BEFORE TEST EVALUATION
# =========================================================


def parameter_json_safe(value):
    if isinstance(value, dict):
        return {
            str(key): parameter_json_safe(item)
            for key, item in value.items()
        }
    if isinstance(value, (list, tuple)):
        return [parameter_json_safe(item) for item in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, Path):
        return str(value)
    return value


PREVIOUS_PAPER_BASELINE_PARAMS = copy.deepcopy(
    FROZEN_BASELINE_PARAMS
)
baseline_parameter_changes = {}
for parameter_name in sorted(
    set(PREVIOUS_PAPER_BASELINE_PARAMS)
    | set(BASELINE_TUNING_SELECTED)
):
    previous_value = parameter_json_safe(
        PREVIOUS_PAPER_BASELINE_PARAMS.get(parameter_name)
    )
    selected_value = parameter_json_safe(
        BASELINE_TUNING_SELECTED.get(parameter_name)
    )
    if previous_value != selected_value:
        baseline_parameter_changes[parameter_name] = {
            "previous": previous_value,
            "five_seed_selected": selected_value,
        }

FROZEN_BASELINE_PARAMS = copy.deepcopy(
    BASELINE_TUNING_SELECTED
)
baseline_parameter_source = (
    "five_seed_calibration_only_all_seed_feasibility"
)

if RUN_TRACE_PARAMETER_SEARCH:
    TRACE_TUNING_SELECTED = {
        "profile_fraction": trace_search_best["profile_fraction"],
        "cp_fraction_within_rest": (
            trace_search_best["cp_fraction_within_rest"]
        ),
        "num_groups": SEARCH_NUM_GROUPS,
        "smooth_window": trace_search_best["smooth_window"],
        "floor_ratio": trace_search_best["floor_ratio"],
        "ridge_alpha": trace_search_best["ridge_alpha"],
        "search_selected_lambda": trace_search_best["lambda_value"],
        "certified_lambda_grid": certified_lambda_grid,
    }
else:
    raise RuntimeError(
        "TRACE parameter search must remain enabled for this notebook."
    )

FROZEN_TRACE_PARAMS = {
    "profile_fraction": 0.10,
    "cp_fraction_within_rest": 0.15,
    "num_groups": 2,
    "smooth_window": 3,
    "floor_ratio": 0.15,
    "ridge_alpha": 1.0,
    "lambda_grid_initial": lambda_grid_initial,
    "lambda_grid_extension": lambda_grid_extension,
    "lambda_grid_master": lambda_grid_master,
    "lambda_max_cap": lambda_grid_max_cap,
    "lambda_expand_step": lambda_grid_expand_step,
    "multiple_testing_method": "holm_stepdown_hb",
    "empirical_fallback_allowed": False,
}

trace_structure_keys = [
    "profile_fraction",
    "cp_fraction_within_rest",
    "num_groups",
    "smooth_window",
    "floor_ratio",
    "ridge_alpha",
]
trace_tuning_differences = {
    key: {
        "tuned": parameter_json_safe(TRACE_TUNING_SELECTED[key]),
        "frozen": parameter_json_safe(FROZEN_TRACE_PARAMS[key]),
    }
    for key in trace_structure_keys
    if parameter_json_safe(TRACE_TUNING_SELECTED[key])
    != parameter_json_safe(FROZEN_TRACE_PARAMS[key])
}
if trace_tuning_differences:
    raise RuntimeError(
        "The TRACE structural search changed unexpectedly: "
        f"{trace_tuning_differences}"
    )

trace_profile_fraction = FROZEN_TRACE_PARAMS["profile_fraction"]
trace_cp_fraction_within_rest = FROZEN_TRACE_PARAMS[
    "cp_fraction_within_rest"
]
trace_num_groups = FROZEN_TRACE_PARAMS["num_groups"]
trace_smooth_window = FROZEN_TRACE_PARAMS["smooth_window"]
trace_floor_ratio = FROZEN_TRACE_PARAMS["floor_ratio"]
trace_ridge_alpha = FROZEN_TRACE_PARAMS["ridge_alpha"]

FROZEN_ABLATION_PARAMS = {
    M_GLOBAL_CRC: {
        "structural_hyperparameters": {},
        "lambda_family": lambda_grid_master,
    },
    M_HORIZON_PROFILE_CRC: {
        "smooth_window": trace_smooth_window,
        "floor_ratio": trace_floor_ratio,
        "lambda_family": lambda_grid_master,
    },
    M_TRAJECTORY_STRATIFIED_CRC: {
        "num_groups": trace_num_groups,
        "ridge_alpha": trace_ridge_alpha,
        "lambda_family": lambda_grid_master,
    },
    M_TRACE_CRC: {
        "profile_fraction": trace_profile_fraction,
        "cp_fraction_within_rest": trace_cp_fraction_within_rest,
        "num_groups": trace_num_groups,
        "smooth_window": trace_smooth_window,
        "floor_ratio": trace_floor_ratio,
        "ridge_alpha": trace_ridge_alpha,
        "lambda_family": lambda_grid_master,
    },
}

PARAMETER_SELECTION_ARTIFACT = {
    "test_data_used_for_tuning": False,
    "baseline_parameter_source": baseline_parameter_source,
    "baseline_tuning_seeds": BASELINE_TUNING_SEEDS,
    "baseline_tuning_outer_split_seed": PRIMARY_SPLIT_SEED,
    "baseline_tuning_train_size_per_seed": (
        baseline_tune_train_size
    ),
    "baseline_tuning_validation_size_per_seed": (
        baseline_tune_validation_size
    ),
    "baseline_seed_aggregation": (
        "seed-level metrics; no pooling across repeated trajectories"
    ),
    "baseline_feasibility_rule": (
        "scope-appropriate coverage target met on every inference seed"
    ),
    "baseline_efficiency_rule": (
        "minimum mean validation AFR among all-seed-feasible candidates"
    ),
    "baseline_no_feasible_candidate_rule": (
        "minimum maximum seed shortfall, then mean shortfall, then AFR"
    ),
    "baseline_selected_validation_summary": {
        method: result["summary"]
        for method, result in baseline_tuning_results.items()
    },
    "baseline_tuning_selected": BASELINE_TUNING_SELECTED,
    "baseline_frozen_for_evaluation": FROZEN_BASELINE_PARAMS,
    "previous_to_five_seed_parameter_changes": (
        baseline_parameter_changes
    ),
    "trace_parameter_selection_changed": False,
    "trace_tuning_selected": TRACE_TUNING_SELECTED,
    "trace_frozen_for_evaluation": FROZEN_TRACE_PARAMS,
    "trace_tuning_to_frozen_differences": (
        trace_tuning_differences
    ),
    "ablation_frozen_parameters": FROZEN_ABLATION_PARAMS,
}

with (robustness_dir / "parameter_selection.pkl").open("wb") as file:
    pickle.dump(
        {
            **PARAMETER_SELECTION_ARTIFACT,
            "baseline_tuning_results": baseline_tuning_results,
            "baseline_tuning_candidates": baseline_tuning_candidates,
        },
        file,
    )
with (robustness_dir / "parameter_selection.json").open("w") as file:
    json.dump(
        parameter_json_safe(PARAMETER_SELECTION_ARTIFACT),
        file,
        indent=2,
    )

print("\nFROZEN PARAMETER MANIFEST")
print("Baseline parameter source:", baseline_parameter_source)
print(
    "Baseline parameters changed from previous paper settings:",
    sorted(baseline_parameter_changes),
)
print("TRACE parameter selection changed: False")
print("Saved parameter selection to:", robustness_dir)


In [ ]:
# =========================================================
# ALL PAPER METHODS: FIXED RANDOM 300/700 SPLIT ACROSS SEEDS
# =========================================================

def run_paper_methods_for_split(
    residual_all,
    predicted_norm_all,
    calibration_indices,
    test_indices,
):
    residual_all = np.asarray(residual_all, dtype=float)
    predicted_norm_all = np.asarray(predicted_norm_all, dtype=float)
    calibration_indices = np.asarray(calibration_indices, dtype=int)
    test_indices = np.asarray(test_indices, dtype=int)

    if np.intersect1d(calibration_indices, test_indices).size:
        raise ValueError("Calibration and test indices must be disjoint.")

    res_cal = residual_all[calibration_indices]
    res_test = residual_all[test_indices]
    pred_cal = predicted_norm_all[calibration_indices]
    pred_test = predicted_norm_all[test_indices]

    n_test, horizon_count = res_test.shape
    results = []

    def add(name, category, radius):
        results.append(
            evaluate_radius_result(name, category, radius, res_test)
        )

    q_constant = conformal_order_stat(res_cal.reshape(-1), alpha)
    add(
        M_GLOBAL_RESIDUAL,
        C_STANDARD,
        np.full_like(res_test, q_constant),
    )

    q_horizon = np.array(
        [
            conformal_order_stat(res_cal[:, horizon], alpha)
            for horizon in range(horizon_count)
        ]
    )
    add(
        M_HORIZON_CONFORMAL,
        C_STANDARD,
        np.tile(q_horizon[None, :], (n_test, 1)),
    )

    q_max_score = conformal_order_stat(
        np.max(res_cal, axis=1),
        alpha,
    )
    add(
        M_MAX_SCORE,
        C_STANDARD,
        np.full_like(res_test, q_max_score),
    )

    cqr_output = run_cqr_style_residual_quantile_cp(
        res_cal,
        res_test,
        alpha,
    )
    add(
        M_RESIDUAL_QUANTILE,
        C_STANDARD,
        cqr_output["radius"],
    )

    add(
        M_WEIGHTED_HORIZON,
        C_TIME,
        run_weighted_cp(
            res_cal,
            res_test,
            alpha,
            FROZEN_BASELINE_PARAMS["weighted_horizon_beta"],
            mode="stepwise",
        ),
    )
    add(
        M_WEIGHTED_TRAJECTORY,
        C_TIME,
        run_weighted_cp(
            res_cal,
            res_test,
            alpha,
            FROZEN_BASELINE_PARAMS["weighted_trajectory_beta"],
            mode="trajectory",
        ),
    )

    enbpi_h = FROZEN_BASELINE_PARAMS["enbpi_horizon"]
    enbpi_h_output = run_enbpi_csi_stepwise(
        fro_res_cal=res_cal,
        fro_res_test=res_test,
        base_model=Ridge(alpha=1.0),
        alpha_level=alpha,
        B=enbpi_h["B"],
        window=enbpi_h["window"],
        online_update=True,
        center_method=enbpi_h["center_method"],
        random_state=123,
    )
    add(M_ENBPI_HORIZON, C_TIME, enbpi_h_output["radius"])

    enbpi_t = FROZEN_BASELINE_PARAMS["enbpi_trajectory"]
    enbpi_t_output = run_enbpi_csi_trajectory(
        fro_res_cal=res_cal,
        fro_res_test=res_test,
        base_model=Ridge(alpha=1.0),
        alpha_level=alpha,
        B=enbpi_t["B"],
        window=enbpi_t["window"],
        online_update=True,
        center_method=enbpi_t["center_method"],
        random_state=456,
    )
    add(M_ENBPI_TRAJECTORY, C_TIME, enbpi_t_output["radius"])

    aci_h_output = run_aci_stepwise(
        res_cal=res_cal,
        res_test=res_test,
        alpha_target=alpha,
        gamma=FROZEN_BASELINE_PARAMS["aci_horizon_gamma"],
        update_history=True,
    )
    add(M_ACI_HORIZON, C_TIME, aci_h_output["radius"])

    aci_t_output = run_aci_trajectory(
        res_cal=res_cal,
        res_test=res_test,
        alpha_target=alpha,
        gamma=FROZEN_BASELINE_PARAMS["aci_trajectory_gamma"],
        update_history=True,
    )
    add(M_ACI_TRAJECTORY, C_TIME, aci_t_output["radius"])

    agaci_h = FROZEN_BASELINE_PARAMS["agaci_horizon"]
    agaci_h_output = run_agaci_stepwise(
        res_cal=res_cal,
        res_test=res_test,
        alpha_target=alpha,
        gammas=agaci_h["gammas"],
        eta=agaci_h["eta"],
        update_history=True,
    )
    add(M_AGACI_HORIZON, C_TIME, agaci_h_output["radius"])

    agaci_t = FROZEN_BASELINE_PARAMS["agaci_trajectory"]
    agaci_t_output = run_agaci_trajectory(
        res_cal=res_cal,
        res_test=res_test,
        alpha_target=alpha,
        gammas=agaci_t["gammas"],
        eta=agaci_t["eta"],
        update_history=True,
    )
    add(M_AGACI_TRAJECTORY, C_TIME, agaci_t_output["radius"])

    alpha_bonferroni = alpha / horizon_count
    q_bonferroni = np.array(
        [
            conformal_order_stat(
                res_cal[:, horizon],
                alpha_bonferroni,
            )
            for horizon in range(horizon_count)
        ]
    )
    add(
        M_BONFERRONI,
        C_STRUCTURED,
        np.tile(q_bonferroni[None, :], (n_test, 1)),
    )

    alpha_sidak = 1.0 - (1.0 - alpha) ** (1.0 / horizon_count)
    q_sidak = np.array(
        [
            conformal_order_stat(res_cal[:, horizon], alpha_sidak)
            for horizon in range(horizon_count)
        ]
    )
    add(
        M_SIDAK,
        C_STRUCTURED,
        np.tile(q_sidak[None, :], (n_test, 1)),
    )

    copula_params = FROZEN_BASELINE_PARAMS["copulacpts"]
    copula_output = run_copulacpts_residual_norm(
        res_cal=res_cal,
        res_test=res_test,
        alpha_level=alpha,
        split=copula_params["split"],
        epochs=copula_params["epochs"],
        random_state=123,
    )
    add(M_COPULACPTS, C_STRUCTURED, copula_output["radius"])

    for method, style in (
        (M_GLOBAL_CRC, "global"),
        (M_HORIZON_PROFILE_CRC, "horizon"),
        (M_TRAJECTORY_STRATIFIED_CRC, "groups"),
        (M_TRACE_CRC, "full"),
    ):
        results.append(
            run_trace_style(
                method,
                style,
                res_cal,
                pred_cal,
                res_test,
                pred_test,
            )
        )

    result_names = [result["method"] for result in results]
    if result_names != METHOD_ORDER:
        raise RuntimeError(
            "Paper-method order mismatch. "
            f"Expected {METHOD_ORDER}, got {result_names}."
        )
    return results


n_total, horizon_count = reference_shape
primary_cal_indices, primary_test_indices = make_trajectory_split(
    n_total,
    PRIMARY_SPLIT_SEED,
)
if len(primary_cal_indices) != CALIBRATION_SIZE:
    raise RuntimeError("Primary calibration split has the wrong size.")

method_seed_rows = []
fixed_coverage_curves = np.zeros(
    (len(INFERENCE_SEEDS), len(METHOD_ORDER), T_out),
    dtype=float,
)
fixed_radius_curves = np.zeros_like(fixed_coverage_curves)

for seed_index, seed in enumerate(INFERENCE_SEEDS):
    compact = seed_compact_data[seed]
    seed_results = run_paper_methods_for_split(
        compact["fro_residual"],
        compact["predicted_norm"],
        primary_cal_indices,
        primary_test_indices,
    )

    for method_index, result in enumerate(seed_results):
        fixed_coverage_curves[seed_index, method_index] = result.pop(
            "per_horizon_coverage"
        )
        fixed_radius_curves[seed_index, method_index] = result.pop(
            "per_horizon_radius"
        )
        result.update(
            {
                "run_id": len(method_seed_rows),
                "inference_seed": int(seed),
                "split": "fixed_random_300_700",
                "split_seed": int(PRIMARY_SPLIT_SEED),
                "n_cal": int(len(primary_cal_indices)),
                "n_test": int(len(primary_test_indices)),
            }
        )
        method_seed_rows.append(result)

    trace_result = next(
        result
        for result in seed_results
        if result["method"] == M_TRACE_CRC
    )
    print(
        f"seed={seed} methods={len(seed_results)} "
        f"TRACE lambda={trace_result['lambda_star']:.4f} "
        f"TRACE TC={trace_result['trajectory_coverage']:.3f} "
        f"TRACE AFR={trace_result['afr']:.3f}"
    )

print(
    "fixed-random-split method/inference-seed evaluations:",
    len(method_seed_rows),
)


In [ ]:
# =========================================================
# TRACE-CRC ONLY: REPEATED RANDOM-SPLIT ROBUSTNESS
# =========================================================

n_total, horizon_count = reference_shape
split_indices_by_seed = {
    int(split_seed): make_trajectory_split(n_total, split_seed)
    for split_seed in ROBUSTNESS_SPLIT_SEEDS
}

trace_split_rows = []
trace_split_coverage_curves = []
trace_split_radius_curves = []

for split_seed in ROBUSTNESS_SPLIT_SEEDS:
    calibration_indices, test_indices = split_indices_by_seed[int(split_seed)]

    for inference_seed in INFERENCE_SEEDS:
        residual_all = seed_compact_data[inference_seed]["fro_residual"]
        predicted_norm_all = seed_compact_data[inference_seed]["predicted_norm"]

        result = run_trace_style(
            M_TRACE_CRC,
            "full",
            residual_all[calibration_indices],
            predicted_norm_all[calibration_indices],
            residual_all[test_indices],
            predicted_norm_all[test_indices],
        )

        trace_split_coverage_curves.append(
            result.pop("per_horizon_coverage")
        )
        trace_split_radius_curves.append(result.pop("per_horizon_radius"))
        result.update(
            {
                "run_id": len(trace_split_rows),
                "inference_seed": int(inference_seed),
                "split_seed": int(split_seed),
                "n_cal": int(len(calibration_indices)),
                "n_test": int(len(test_indices)),
                "analysis": "TRACE-only repeated random trajectory split",
            }
        )
        trace_split_rows.append(result)

        print(
            f"TRACE inference_seed={inference_seed} "
            f"split_seed={split_seed} "
            f"lambda={result['lambda_star']:.4f} "
            f"formal={result['formal']} "
            f"TC={result['trajectory_coverage']:.3f} "
            f"AFR={result['afr']:.3f}"
        )

trace_split_coverage_curves = np.asarray(
    trace_split_coverage_curves,
    dtype=float,
)
trace_split_radius_curves = np.asarray(
    trace_split_radius_curves,
    dtype=float,
)

expected_trace_runs = len(INFERENCE_SEEDS) * len(ROBUSTNESS_SPLIT_SEEDS)
if len(trace_split_rows) != expected_trace_runs:
    raise RuntimeError("TRACE repeated-split run count is incorrect.")

print("TRACE seed/split evaluations:", len(trace_split_rows))


In [ ]:
# =========================================================
# AGGREGATE AND SAVE BOTH ROBUSTNESS ANALYSES
# =========================================================


def write_csv(path, rows, fieldnames=None):
    rows = list(rows)
    if not rows:
        raise ValueError(f"No rows available for {path}.")
    if fieldnames is None:
        fieldnames = []
        seen = set()
        for row in rows:
            for key in row:
                if key not in seen:
                    fieldnames.append(key)
                    seen.add(key)
    with Path(path).open("w", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def json_safe(value):
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(item) for item in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, Path):
        return str(value)
    return value


def csv_safe_rows(rows):
    output = []
    for row in rows:
        clean = dict(row)
        clean["accepted_lambdas"] = json.dumps(
            clean.get("accepted_lambdas", [])
        )
        for grid_key in (
            "lambda_grid_initial",
            "lambda_grid_extension",
            "lambda_grid_master",
            "lambda_grid_used",
        ):
            clean[grid_key] = json.dumps(
                json_safe(clean.get(grid_key, []))
            )
        output.append(clean)
    return output


SUMMARY_METRICS = [
    ("mhc", "MHC"),
    ("whc", "WHC"),
    ("trajectory_coverage", "TC"),
    ("afr", "AFR"),
    ("maximum_horizon_afr", "Maximum horizon AFR"),
    ("lambda_star", "Selected lambda"),
]


def numeric_summary(values):
    values = np.asarray(
        [value for value in values if value is not None],
        dtype=float,
    )
    if len(values) == 0:
        return {
            "mean": None,
            "std": None,
            "median": None,
            "min": None,
            "max": None,
        }
    return {
        "mean": float(np.mean(values)),
        "std": float(np.std(values)),
        "median": float(np.median(values)),
        "min": float(np.min(values)),
        "max": float(np.max(values)),
    }


method_seed_summary = []
for method in METHOD_ORDER:
    rows = [row for row in method_seed_rows if row["method"] == method]
    summary_row = {
        "category": METHOD_CATEGORY[method],
        "method": method,
        "display_name": DISPLAY_NAME[method],
        "n_inference_seeds": len(rows),
    }
    for key, _ in SUMMARY_METRICS:
        stats = numeric_summary([row[key] for row in rows])
        for statistic, value in stats.items():
            summary_row[f"{key}_{statistic}"] = value
    formal_values = [
        row["formal"]
        for row in rows
        if row["formal"] is not None
    ]
    summary_row["formal_rate"] = (
        float(np.mean(formal_values)) if formal_values else None
    )
    method_seed_summary.append(summary_row)


trace_split_summary = {}
for key, label in SUMMARY_METRICS:
    trace_split_summary[key] = {
        "label": label,
        **numeric_summary([row[key] for row in trace_split_rows]),
    }
trace_split_summary["formal_rate"] = float(
    np.mean([row["formal"] for row in trace_split_rows])
)
trace_split_summary["coverage_target_rate"] = float(
    np.mean(
        [
            row["trajectory_coverage"] >= 1.0 - alpha
            for row in trace_split_rows
        ]
    )
)


FACTOR_SENSITIVITY_METRICS = SUMMARY_METRICS + [
    ("formal_indicator", "Formal selection rate"),
    ("coverage_target_indicator", "TC target pass rate"),
]


def trace_factor_value(row, key):
    if key == "formal_indicator":
        return float(row["formal"])
    if key == "coverage_target_indicator":
        return float(row["trajectory_coverage"] >= 1.0 - alpha)
    return float(row[key])


def summarize_trace_levels(group_key, levels):
    summaries = []
    for level in levels:
        rows = [
            row for row in trace_split_rows
            if int(row[group_key]) == int(level)
        ]
        summary = {
            group_key: int(level),
            "n_runs": len(rows),
        }
        for key, _ in FACTOR_SENSITIVITY_METRICS:
            values = [trace_factor_value(row, key) for row in rows]
            summary[f"{key}_mean"] = float(np.mean(values))
            summary[f"{key}_std"] = float(np.std(values))
        summaries.append(summary)
    return summaries


trace_by_inference_seed = summarize_trace_levels(
    "inference_seed",
    INFERENCE_SEEDS,
)
trace_by_split_seed = summarize_trace_levels(
    "split_seed",
    ROBUSTNESS_SPLIT_SEEDS,
)

trace_factor_sensitivity = []
for key, label in FACTOR_SENSITIVITY_METRICS:
    all_values = np.asarray(
        [trace_factor_value(row, key) for row in trace_split_rows],
        dtype=float,
    )
    inference_marginal_means = np.asarray(
        [row[f"{key}_mean"] for row in trace_by_inference_seed],
        dtype=float,
    )
    split_marginal_means = np.asarray(
        [row[f"{key}_mean"] for row in trace_by_split_seed],
        dtype=float,
    )
    trace_factor_sensitivity.append(
        {
            "metric": key,
            "label": label,
            "overall_mean": float(np.mean(all_values)),
            "overall_std": float(np.std(all_values)),
            "inference_seed_marginal_mean": float(
                np.mean(inference_marginal_means)
            ),
            "inference_seed_marginal_std": float(
                np.std(inference_marginal_means)
            ),
            "inference_seed_marginal_min": float(
                np.min(inference_marginal_means)
            ),
            "inference_seed_marginal_max": float(
                np.max(inference_marginal_means)
            ),
            "split_seed_marginal_mean": float(
                np.mean(split_marginal_means)
            ),
            "split_seed_marginal_std": float(
                np.std(split_marginal_means)
            ),
            "split_seed_marginal_min": float(
                np.min(split_marginal_means)
            ),
            "split_seed_marginal_max": float(
                np.max(split_marginal_means)
            ),
        }
    )


steps = np.arange(1, T_out + 1)
method_seed_per_horizon = []
for seed_index, inference_seed in enumerate(INFERENCE_SEEDS):
    for method_index, method in enumerate(METHOD_ORDER):
        for horizon_index, step in enumerate(steps):
            method_seed_per_horizon.append(
                {
                    "inference_seed": int(inference_seed),
                    "split_seed": int(PRIMARY_SPLIT_SEED),
                    "category": METHOD_CATEGORY[method],
                    "method": method,
                    "display_name": DISPLAY_NAME[method],
                    "step": int(step),
                    "coverage": float(
                        fixed_coverage_curves[
                            seed_index,
                            method_index,
                            horizon_index,
                        ]
                    ),
                    "mean_radius": float(
                        fixed_radius_curves[
                            seed_index,
                            method_index,
                            horizon_index,
                        ]
                    ),
                }
            )

trace_split_per_horizon = []
for run_index, row in enumerate(trace_split_rows):
    for horizon_index, step in enumerate(steps):
        trace_split_per_horizon.append(
            {
                "run_id": row["run_id"],
                "inference_seed": row["inference_seed"],
                "split_seed": row["split_seed"],
                "step": int(step),
                "coverage": float(
                    trace_split_coverage_curves[run_index, horizon_index]
                ),
                "mean_radius": float(
                    trace_split_radius_curves[run_index, horizon_index]
                ),
            }
        )

baseline_tuning_candidate_rows = []
baseline_tuning_seed_rows = []
for method_name, candidates in baseline_tuning_candidates.items():
    selected_candidate = baseline_tuning_results[method_name]
    for candidate_index, candidate in enumerate(candidates):
        summary = candidate["summary"]
        selected = candidate is selected_candidate
        baseline_tuning_candidate_rows.append(
            {
                "method": method_name,
                "candidate_index": candidate_index,
                "selected": selected,
                "parameters_json": json.dumps(
                    json_safe(candidate["params"]),
                    sort_keys=True,
                ),
                **summary,
                "selection_key_json": json.dumps(
                    json_safe(candidate["key"])
                ),
            }
        )
        for seed_metrics in candidate["seed_metrics"]:
            baseline_tuning_seed_rows.append(
                {
                    "method": method_name,
                    "candidate_index": candidate_index,
                    "selected": selected,
                    "parameters_json": json.dumps(
                        json_safe(candidate["params"]),
                        sort_keys=True,
                    ),
                    **seed_metrics,
                }
            )

write_csv(
    table_dir / "baseline_tuning_candidates.csv",
    baseline_tuning_candidate_rows,
)
write_csv(
    table_dir / "baseline_tuning_seed_metrics.csv",
    baseline_tuning_seed_rows,
)

write_csv(
    table_dir / "method_inference_seed_metrics.csv",
    csv_safe_rows(method_seed_rows),
)
write_csv(
    table_dir / "method_inference_seed_summary.csv",
    method_seed_summary,
)
write_csv(
    table_dir / "method_inference_seed_per_horizon.csv",
    method_seed_per_horizon,
)
write_csv(
    table_dir / "trace_crc_split_metrics.csv",
    csv_safe_rows(trace_split_rows),
)
write_csv(
    table_dir / "trace_crc_split_per_horizon.csv",
    trace_split_per_horizon,
)
write_csv(
    table_dir / "trace_crc_by_inference_seed.csv",
    trace_by_inference_seed,
)
write_csv(
    table_dir / "trace_crc_by_split_seed.csv",
    trace_by_split_seed,
)
write_csv(
    table_dir / "trace_crc_factor_sensitivity.csv",
    trace_factor_sensitivity,
)

robustness_cal_indices = np.stack(
    [
        split_indices_by_seed[int(split_seed)][0]
        for split_seed in ROBUSTNESS_SPLIT_SEEDS
    ]
)
robustness_test_indices = np.stack(
    [
        split_indices_by_seed[int(split_seed)][1]
        for split_seed in ROBUSTNESS_SPLIT_SEEDS
    ]
)

config = {
    "inference_seeds": INFERENCE_SEEDS,
    "primary_outer_split": {
        "type": "fixed random trajectory-level holdout",
        "split_seed": PRIMARY_SPLIT_SEED,
        "calibration_size": int(len(primary_cal_indices)),
        "test_size": int(len(primary_test_indices)),
        "calibration_fraction": CALIBRATION_FRACTION,
        "applies_to": "all paper methods",
    },
    "trace_split_robustness": {
        "type": "repeated random trajectory-level holdout",
        "split_seeds": ROBUSTNESS_SPLIT_SEEDS,
        "n_splits": len(ROBUSTNESS_SPLIT_SEEDS),
        "calibration_size": CALIBRATION_SIZE,
        "test_size": n_total - CALIBRATION_SIZE,
        "applies_to": M_TRACE_CRC,
    },
    "trace_internal_partition": {
        "assignment": (
            "disjoint slices of the randomly permuted calibration indices"
        ),
        "profile_fraction": trace_profile_fraction,
        "cp_fraction_within_rest": trace_cp_fraction_within_rest,
        "n_prof": int(round(trace_profile_fraction * CALIBRATION_SIZE)),
        "n_cp": int(
            round(
                trace_cp_fraction_within_rest
                * (
                    CALIBRATION_SIZE
                    - round(trace_profile_fraction * CALIBRATION_SIZE)
                )
            )
        ),
    },
    "alpha": alpha,
    "delta_ltt": delta_ltt,
    "trace_num_groups": trace_num_groups,
    "trace_smooth_window": trace_smooth_window,
    "trace_floor_ratio": trace_floor_ratio,
    "trace_ridge_alpha": trace_ridge_alpha,
    "lambda_grid": lambda_grid_master.tolist(),
    "lambda_grid_initial": lambda_grid_initial.tolist(),
    "lambda_grid_extension": lambda_grid_extension.tolist(),
    "lambda_grid_master": lambda_grid_master.tolist(),
    "lambda_family_size": int(len(lambda_grid_master)),
    "lambda_initial_max": float(lambda_grid_initial[-1]),
    "lambda_max_cap": float(lambda_grid_max_cap),
    "lambda_expand_step": float(lambda_grid_expand_step),
    "lambda_expansion_policy": (
        "prespecified_master_grid_joint_holm_hb"
    ),
    "multiple_testing_method": "holm_stepdown_hb",
    "empirical_fallback_allowed": False,
    "global_crc": {
        "use_all_calibration_for_ltt": use_all_calib_global_crc,
    },
    "risk_control_data_provenance": {
        M_GLOBAL_CRC: {
            "D_prof": None,
            "D_cp": "pooled global scale",
            "D_val": "trajectory-level LTT multiplier",
        },
        M_HORIZON_PROFILE_CRC: {
            "D_prof": "horizon profile",
            "D_cp": "pooled global scale",
            "D_val": "trajectory-level LTT multiplier",
        },
        M_TRAJECTORY_STRATIFIED_CRC: {
            "D_prof": "raw difficulty model",
            "D_cp": "raw group threshold and quantiles",
            "D_val": "trajectory-level LTT multiplier",
        },
        M_TRACE_CRC: {
            "D_prof": "horizon profile and normalized difficulty model",
            "D_cp": "normalized group threshold and quantiles",
            "D_val": "trajectory-level LTT multiplier",
        },
    },
    "frozen_baseline_params": FROZEN_BASELINE_PARAMS,
    "frozen_trace_params": FROZEN_TRACE_PARAMS,
    "frozen_ablation_params": FROZEN_ABLATION_PARAMS,
    "parameter_selection": PARAMETER_SELECTION_ARTIFACT,
    "method_order": METHOD_ORDER,
    "display_names": DISPLAY_NAME,
    "method_colors": METHOD_COLORS,
    "figure_2_pair_colors": FIGURE_2_PAIR_COLORS,
    "excluded_paper_methods": EXCLUDED_PAPER_METHODS,
    "aggregate_interpretation": (
        "Descriptive across reused trajectories and inference seeds; "
        "not independent replications or a simultaneous certificate."
    ),
}
config["trace_internal_partition"]["n_ltt"] = int(
    CALIBRATION_SIZE
    - config["trace_internal_partition"]["n_prof"]
    - config["trace_internal_partition"]["n_cp"]
)

robustness_outputs = {
    "config": config,
    "method_seed_rows": method_seed_rows,
    "method_seed_summary": method_seed_summary,
    "trace_split_rows": trace_split_rows,
    "trace_split_summary": trace_split_summary,
    "trace_by_inference_seed": trace_by_inference_seed,
    "trace_by_split_seed": trace_by_split_seed,
    "trace_factor_sensitivity": trace_factor_sensitivity,
    "fixed_coverage_curves": fixed_coverage_curves,
    "fixed_radius_curves": fixed_radius_curves,
    "trace_split_coverage_curves": trace_split_coverage_curves,
    "trace_split_radius_curves": trace_split_radius_curves,
    "primary_cal_indices": primary_cal_indices,
    "primary_test_indices": primary_test_indices,
    "robustness_cal_indices": robustness_cal_indices,
    "robustness_test_indices": robustness_test_indices,
    "parameter_selection": PARAMETER_SELECTION_ARTIFACT,
    "baseline_tuning_results": baseline_tuning_results,
    "baseline_tuning_candidates": baseline_tuning_candidates,
}
with (robustness_dir / "robustness_outputs.pkl").open("wb") as file:
    pickle.dump(robustness_outputs, file)

np.savez_compressed(
    robustness_dir / "robustness_arrays.npz",
    fixed_coverage_curves=fixed_coverage_curves,
    fixed_radius_curves=fixed_radius_curves,
    trace_split_coverage_curves=trace_split_coverage_curves,
    trace_split_radius_curves=trace_split_radius_curves,
    inference_seeds=np.asarray(INFERENCE_SEEDS, dtype=int),
    robustness_split_seeds=np.asarray(
        ROBUSTNESS_SPLIT_SEEDS,
        dtype=int,
    ),
    method_order=np.asarray(METHOD_ORDER, dtype=str),
    primary_cal_indices=primary_cal_indices,
    primary_test_indices=primary_test_indices,
    robustness_cal_indices=robustness_cal_indices,
    robustness_test_indices=robustness_test_indices,
    steps=steps,
)

summary_payload = {
    "config": config,
    "method_seed_run_count": len(method_seed_rows),
    "trace_split_run_count": len(trace_split_rows),
    "method_seed_summary": method_seed_summary,
    "trace_split_summary": trace_split_summary,
    "trace_by_inference_seed": trace_by_inference_seed,
    "trace_by_split_seed": trace_by_split_seed,
    "trace_factor_sensitivity": trace_factor_sensitivity,
    "parameter_selection": PARAMETER_SELECTION_ARTIFACT,
}
with (robustness_dir / "robustness_summary.json").open("w") as file:
    json.dump(json_safe(summary_payload), file, indent=2)

manifest = {
    "output_root": robustness_dir.relative_to(PROJECT_ROOT).as_posix(),
    "files": {
        "pickle": "robustness_outputs.pkl",
        "arrays": "robustness_arrays.npz",
        "summary_json": "robustness_summary.json",
        "parameter_selection_pickle": "parameter_selection.pkl",
        "parameter_selection_json": "parameter_selection.json",
        "baseline_tuning_candidates": "tables/baseline_tuning_candidates.csv",
        "baseline_tuning_seed_metrics": "tables/baseline_tuning_seed_metrics.csv",
        "method_seed_metrics": (
            "tables/method_inference_seed_metrics.csv"
        ),
        "method_seed_summary": (
            "tables/method_inference_seed_summary.csv"
        ),
        "method_seed_per_horizon": (
            "tables/method_inference_seed_per_horizon.csv"
        ),
        "trace_split_metrics": "tables/trace_crc_split_metrics.csv",
        "trace_split_per_horizon": (
            "tables/trace_crc_split_per_horizon.csv"
        ),
        "trace_by_inference_seed": (
            "tables/trace_crc_by_inference_seed.csv"
        ),
        "trace_by_split_seed": "tables/trace_crc_by_split_seed.csv",
        "trace_factor_sensitivity": (
            "tables/trace_crc_factor_sensitivity.csv"
        ),
    },
}
with (robustness_dir / "manifest.json").open("w") as file:
    json.dump(manifest, file, indent=2)

print("Saved robustness artifacts to:", robustness_dir)


In [ ]:
# =========================================================
# PLAIN-TEXT ROBUSTNESS TABLES + SILENT LATEX EXPORT
# =========================================================

print("\nALL METHODS: FIXED RANDOM 300/700 SPLIT ACROSS INFERENCE SEEDS")
print(
    f"{'Category':<27} {'Method':<31} "
    f"{'MHC':>15} {'WHC':>15} {'TC':>15} {'AFR':>15}"
)
print("-" * 124)
for row in method_seed_summary:
    print(
        f"{row['category']:<27} "
        f"{row['display_name']:<31} "
        f"{row['mhc_mean']:.3f} +/- {row['mhc_std']:.3f} "
        f"{row['whc_mean']:.3f} +/- {row['whc_std']:.3f} "
        f"{row['trajectory_coverage_mean']:.3f} +/- "
        f"{row['trajectory_coverage_std']:.3f} "
        f"{row['afr_mean']:.2f} +/- {row['afr_std']:.2f}"
    )

print("\nTRACE-CRC ONLY: INFERENCE SEED x RANDOM TRAJECTORY SPLIT")
print(f"Runs: {len(trace_split_rows)}")
print(f"Formal selection rate: {trace_split_summary['formal_rate']:.3f}")
print(
    "Trajectory target pass rate: "
    f"{trace_split_summary['coverage_target_rate']:.3f}"
)
print(
    f"{'Metric':<25} {'Mean':>10} {'Std':>10} "
    f"{'Median':>10} {'Min':>10} {'Max':>10}"
)
print("-" * 80)
for key, _ in SUMMARY_METRICS:
    item = trace_split_summary[key]
    print(
        f"{item['label']:<25} "
        f"{item['mean']:>10.4f} "
        f"{item['std']:>10.4f} "
        f"{item['median']:>10.4f} "
        f"{item['min']:>10.4f} "
        f"{item['max']:>10.4f}"
    )

print("\nTRACE-CRC VARIABILITY BY RANDOMNESS SOURCE")
print(
    f"{'Metric':<25} {'Overall mean +/- SD':>23} "
    f"{'Inference-seed SD':>18} {'Split-seed SD':>15}"
)
print("-" * 86)
for row in trace_factor_sensitivity:
    print(
        f"{row['label']:<25} "
        f"{row['overall_mean']:>10.4f} +/- "
        f"{row['overall_std']:<8.4f} "
        f"{row['inference_seed_marginal_std']:>18.4f} "
        f"{row['split_seed_marginal_std']:>15.4f}"
    )


def latex_escape(text):
    return (
        str(text)
        .replace("&", r"\&")
        .replace("%", r"\%")
        .replace("_", r"\_")
    )


method_latex = [
    r"\begin{table}[!htbp]",
    r"\centering",
    r"\scriptsize",
    (
        r"\caption{Robustness of all paper-reported methods across diffusion "
        r"inference seeds under one fixed random trajectory-level 300/700 "
        r"split. Values are mean $\pm$ standard deviation across seeds.}"
    ),
    r"\label{tab:all-method-seed-robustness}",
    r"\begin{tabular}{llcccc}",
    r"\toprule",
    r"Category & Method & MHC & WHC & TC & AFR \\",
    r"\midrule",
]
previous_category = None
for row in method_seed_summary:
    category = (
        latex_escape(row["category"])
        if row["category"] != previous_category
        else ""
    )
    previous_category = row["category"]
    method_latex.append(
        f"{category} & {latex_escape(row['display_name'])} & "
        f"${row['mhc_mean']:.3f} \\pm {row['mhc_std']:.3f}$ & "
        f"${row['whc_mean']:.3f} \\pm {row['whc_std']:.3f}$ & "
        f"${row['trajectory_coverage_mean']:.3f} \\pm "
        f"{row['trajectory_coverage_std']:.3f}$ & "
        f"${row['afr_mean']:.2f} \\pm {row['afr_std']:.2f}$ \\\\"
    )
method_latex.extend(
    [
        r"\bottomrule",
        r"\end{tabular}",
        r"\end{table}",
    ]
)
method_latex_path = table_dir / "all_method_seed_robustness_table.tex"
method_latex_path.write_text("\n".join(method_latex) + "\n")

trace_latex = [
    r"\begin{table}[!htbp]",
    r"\centering",
    r"\scriptsize",
    (
        r"\caption{TRACE-CRC robustness across five diffusion inference "
        r"seeds and ten fixed random trajectory-level 300/700 splits. "
        r"Results summarize 50 separately calibrated evaluations.}"
    ),
    r"\label{tab:trace-seed-split-robustness}",
    r"\begin{tabular}{lccccc}",
    r"\toprule",
    r"Metric & Mean $\pm$ SD & Median & Min & Max \\",
    r"\midrule",
]
for key, _ in SUMMARY_METRICS:
    item = trace_split_summary[key]
    trace_latex.append(
        f"{latex_escape(item['label'])} & "
        f"${item['mean']:.4f} \\pm {item['std']:.4f}$ & "
        f"{item['median']:.4f} & {item['min']:.4f} & "
        f"{item['max']:.4f} \\\\"
    )
trace_latex.extend(
    [
        r"\midrule",
        (
            "Formal selection rate & "
            f"{trace_split_summary['formal_rate']:.3f} & -- & -- & -- \\\\"
        ),
        (
            "Trajectory target pass rate & "
            f"{trace_split_summary['coverage_target_rate']:.3f} "
            "& -- & -- & -- \\\\"
        ),
        r"\bottomrule",
        r"\end{tabular}",
        r"\end{table}",
    ]
)
trace_latex_path = table_dir / "trace_crc_seed_split_robustness_table.tex"
trace_latex_path.write_text("\n".join(trace_latex) + "\n")

factor_latex = [
    r"\begin{table}[!htbp]",
    r"\centering",
    r"\scriptsize",
    (
        r"\caption{TRACE-CRC sensitivity to diffusion inference seeds and "
        r"random trajectory-level splits. Continuous metrics are reported "
        r"as mean $\pm$ standard deviation across all 50 evaluations; "
        r"formal-selection and target-pass entries are rates. Inference-seed "
        r"SD is computed across five marginal means, each averaged over ten "
        r"splits; split-seed SD is computed across ten marginal means, each "
        r"averaged over five inference seeds.}"
    ),
    r"\label{tab:trace-factor-sensitivity}",
    r"\begin{tabular}{lccc}",
    r"\toprule",
    (
        r"Metric & Overall mean $\pm$ SD & "
        r"Inference-seed SD & Split-seed SD \\"
    ),
    r"\midrule",
]
for row in trace_factor_sensitivity:
    if row["metric"] in {
        "formal_indicator",
        "coverage_target_indicator",
    }:
        overall_cell = f"{row['overall_mean']:.4f}"
    else:
        overall_cell = (
            f"${row['overall_mean']:.4f} "
            f"\\pm {row['overall_std']:.4f}$"
        )
    factor_latex.append(
        f"{latex_escape(row['label'])} & {overall_cell} & "
        f"{row['inference_seed_marginal_std']:.4f} & "
        f"{row['split_seed_marginal_std']:.4f} \\\\"
    )
factor_latex.extend(
    [
        r"\bottomrule",
        r"\end{tabular}",
        r"\end{table}",
    ]
)
factor_latex_path = table_dir / "trace_crc_factor_sensitivity_table.tex"
factor_latex_path.write_text("\n".join(factor_latex) + "\n")

manifest["files"].update(
    {
        "method_seed_latex": (
            "tables/all_method_seed_robustness_table.tex"
        ),
        "trace_split_latex": (
            "tables/trace_crc_seed_split_robustness_table.tex"
        ),
        "trace_factor_sensitivity_latex": (
            "tables/trace_crc_factor_sensitivity_table.tex"
        ),
    }
)
with (robustness_dir / "manifest.json").open("w") as file:
    json.dump(manifest, file, indent=2)

print("\nSaved LaTeX tables to:", table_dir)


In [ ]:
# =========================================================
# PAPER-CATEGORY FIGURES AND ROBUSTNESS DIAGNOSTICS
# =========================================================

plt.rcParams["svg.fonttype"] = "none"

coverage_mean = fixed_coverage_curves.mean(axis=0)
coverage_std = fixed_coverage_curves.std(axis=0)
radius_mean = fixed_radius_curves.mean(axis=0)
radius_std = fixed_radius_curves.std(axis=0)
method_to_index = {
    method: index
    for index, method in enumerate(METHOD_ORDER)
}

category_file_stems = {
    C_STANDARD: "figure_1_standard_split_conformal_seed_robustness",
    C_TIME: "figure_2_time_series_adaptive_seed_robustness",
    C_STRUCTURED: "figure_3_structured_multistep_seed_robustness",
    C_RISK: "figure_4_risk_control_ablations_seed_robustness",
}

def paper_line_style(method_name, category):
    if method_name == M_TRACE_CRC:
        return "-", "D", 2.7

    if category == C_TIME:
        if method_name in FIGURE_2_DASHED_METHODS:
            return "--", "o", 2.0
        return "-", "s", 2.0

    return "-", "o", 2.0


for category, methods in METHOD_GROUPS.items():
    figure, axes = plt.subplots(1, 2, figsize=(13.0, 4.8), sharex=True)
    plot_methods = list(methods)
    if M_TRACE_CRC not in plot_methods:
        plot_methods.append(M_TRACE_CRC)

    for method in plot_methods:
        method_index = method_to_index[method]
        color = (
            FIGURE_2_METHOD_COLORS.get(method, METHOD_COLORS[method])
            if category == C_TIME
            else METHOD_COLORS[method]
        )
        label = DISPLAY_NAME[method]
        linestyle, marker, linewidth = paper_line_style(method, category)

        axes[0].plot(
            steps,
            coverage_mean[method_index],
            color=color,
            linestyle=linestyle,
            marker=marker,
            markersize=5,
            linewidth=linewidth,
            label=label,
        )
        axes[0].fill_between(
            steps,
            np.maximum(
                coverage_mean[method_index] - coverage_std[method_index],
                0.0,
            ),
            np.minimum(
                coverage_mean[method_index] + coverage_std[method_index],
                1.0,
            ),
            color=color,
            alpha=0.10,
        )

        axes[1].plot(
            steps,
            radius_mean[method_index],
            color=color,
            linestyle=linestyle,
            marker=marker,
            markersize=5,
            linewidth=linewidth,
            label=label,
        )
        axes[1].fill_between(
            steps,
            np.maximum(
                radius_mean[method_index] - radius_std[method_index],
                0.0,
            ),
            radius_mean[method_index] + radius_std[method_index],
            color=color,
            alpha=0.10,
        )

    axes[0].axhline(
        1.0 - alpha,
        color="black",
        linestyle=":",
        linewidth=1.4,
        label=f"Target {1.0 - alpha:.2f}",
    )
    axes[0].set_title(f"{category}: horizon coverage")
    axes[0].set_ylabel(r"$\mathrm{HC}_j$")
    axes[0].set_ylim(0.65, 1.01)
    axes[1].set_title(f"{category}: Frobenius radius")
    axes[1].set_ylabel(r"$\mathrm{AFR}_j$")

    for axis in axes:
        axis.set_xlabel("Prediction step")
        axis.set_xlim(steps.min(), steps.max())
        axis.grid(True, alpha=0.25)
        axis.legend(fontsize=7, ncol=2)

    figure.tight_layout()
    stem = category_file_stems[category]
    figure.savefig(
        figure_dir / f"{stem}.svg",
        format="svg",
        bbox_inches="tight",
    )
    figure.savefig(
        figure_dir / f"{stem}.pdf",
        bbox_inches="tight",
    )
    plt.show()
    plt.close(figure)


tc_matrix = np.zeros((len(METHOD_ORDER), len(INFERENCE_SEEDS)), dtype=float)
afr_matrix = np.zeros_like(tc_matrix)
for method_index, method in enumerate(METHOD_ORDER):
    for seed_index, seed in enumerate(INFERENCE_SEEDS):
        row = next(
            item
            for item in method_seed_rows
            if item["method"] == method and item["inference_seed"] == seed
        )
        tc_matrix[method_index, seed_index] = row["trajectory_coverage"]
        afr_matrix[method_index, seed_index] = row["afr"]

figure, axes = plt.subplots(1, 2, figsize=(13.5, 8.0))
for axis, matrix, title, color_map in (
    (axes[0], tc_matrix, "Trajectory coverage", "viridis"),
    (axes[1], afr_matrix, "Average Frobenius radius", "magma"),
):
    image = axis.imshow(matrix, aspect="auto", cmap=color_map)
    axis.set_title(title)
    axis.set_xlabel("Diffusion inference seed")
    axis.set_xticks(np.arange(len(INFERENCE_SEEDS)))
    axis.set_xticklabels(
        [str(seed) for seed in INFERENCE_SEEDS],
        rotation=35,
        ha="right",
    )
    axis.set_yticks(np.arange(len(METHOD_ORDER)))
    axis.set_yticklabels([DISPLAY_NAME[method] for method in METHOD_ORDER])
    figure.colorbar(image, ax=axis, fraction=0.046, pad=0.04)

figure.tight_layout()
figure.savefig(
    figure_dir / "all_method_seed_heatmaps.svg",
    format="svg",
    bbox_inches="tight",
)
figure.savefig(
    figure_dir / "all_method_seed_heatmaps.pdf",
    bbox_inches="tight",
)
plt.show()
plt.close(figure)


trace_coverage_mean = trace_split_coverage_curves.mean(axis=0)
trace_coverage_std = trace_split_coverage_curves.std(axis=0)
trace_radius_mean = trace_split_radius_curves.mean(axis=0)
trace_radius_std = trace_split_radius_curves.std(axis=0)

figure, axes = plt.subplots(1, 2, figsize=(12.5, 4.5), sharex=True)
axes[0].plot(
    steps,
    trace_coverage_mean,
    color=METHOD_COLORS[M_TRACE_CRC],
    marker="D",
    linewidth=2.3,
    label="TRACE-CRC mean",
)
axes[0].fill_between(
    steps,
    np.maximum(trace_coverage_mean - trace_coverage_std, 0.0),
    np.minimum(trace_coverage_mean + trace_coverage_std, 1.0),
    color=METHOD_COLORS[M_TRACE_CRC],
    alpha=0.18,
    label="Seed/split +/- 1 SD",
)
axes[0].axhline(
    1.0 - alpha,
    color="black",
    linestyle=":",
    linewidth=1.4,
    label=f"Target {1.0 - alpha:.2f}",
)
axes[0].set_title("TRACE random-split coverage robustness")
axes[0].set_ylabel(r"$\mathrm{HC}_j$")
axes[0].set_ylim(0.65, 1.01)

axes[1].plot(
    steps,
    trace_radius_mean,
    color=METHOD_COLORS[M_TRACE_CRC],
    marker="D",
    linewidth=2.3,
    label="TRACE-CRC mean",
)
axes[1].fill_between(
    steps,
    np.maximum(trace_radius_mean - trace_radius_std, 0.0),
    trace_radius_mean + trace_radius_std,
    color=METHOD_COLORS[M_TRACE_CRC],
    alpha=0.18,
    label="Seed/split +/- 1 SD",
)
axes[1].set_title("TRACE random-split radius robustness")
axes[1].set_ylabel(r"$\mathrm{AFR}_j$")

for axis in axes:
    axis.set_xlabel("Prediction step")
    axis.set_xlim(steps.min(), steps.max())
    axis.grid(True, alpha=0.25)
    axis.legend(fontsize=8)

figure.tight_layout()
figure.savefig(
    figure_dir / "trace_crc_seed_split_curves.svg",
    format="svg",
    bbox_inches="tight",
)
figure.savefig(
    figure_dir / "trace_crc_seed_split_curves.pdf",
    bbox_inches="tight",
)
plt.show()
plt.close(figure)


trace_tc_matrix = np.full(
    (len(INFERENCE_SEEDS), len(ROBUSTNESS_SPLIT_SEEDS)),
    np.nan,
)
trace_afr_matrix = np.full_like(trace_tc_matrix, np.nan)
seed_to_row = {
    seed: index
    for index, seed in enumerate(INFERENCE_SEEDS)
}
for row in trace_split_rows:
    seed_index = seed_to_row[row["inference_seed"]]
    split_index = ROBUSTNESS_SPLIT_SEEDS.index(row["split_seed"])
    trace_tc_matrix[seed_index, split_index] = row["trajectory_coverage"]
    trace_afr_matrix[seed_index, split_index] = row["afr"]

figure, axes = plt.subplots(1, 2, figsize=(13.0, 4.8))
for axis, matrix, title, color_map in (
    (axes[0], trace_tc_matrix, "TRACE trajectory coverage", "viridis"),
    (axes[1], trace_afr_matrix, "TRACE average radius", "magma"),
):
    image = axis.imshow(matrix, aspect="auto", cmap=color_map)
    axis.set_title(title)
    axis.set_xlabel("Random trajectory split")
    axis.set_ylabel("Diffusion inference seed")
    axis.set_xticks(np.arange(len(ROBUSTNESS_SPLIT_SEEDS)))
    axis.set_xticklabels([f"S{i + 1}" for i in range(len(ROBUSTNESS_SPLIT_SEEDS))])
    axis.set_yticks(np.arange(len(INFERENCE_SEEDS)))
    axis.set_yticklabels([str(seed) for seed in INFERENCE_SEEDS])
    figure.colorbar(image, ax=axis, fraction=0.046, pad=0.04)

figure.tight_layout()
figure.savefig(
    figure_dir / "trace_crc_seed_split_heatmaps.svg",
    format="svg",
    bbox_inches="tight",
)
figure.savefig(
    figure_dir / "trace_crc_seed_split_heatmaps.pdf",
    bbox_inches="tight",
)
plt.show()
plt.close(figure)

for obsolete_key in (
    "figure_1_png",
    "figure_2_png",
    "figure_3_png",
    "figure_4_png",
    "all_method_heatmap_png",
    "trace_split_curves_png",
    "trace_split_heatmap_png",
):
    manifest["files"].pop(obsolete_key, None)

manifest["files"].update(
    {
        "figure_1_svg": (
            "figures/figure_1_standard_split_conformal_seed_robustness.svg"
        ),
        "figure_1_pdf": (
            "figures/figure_1_standard_split_conformal_seed_robustness.pdf"
        ),
        "figure_2_svg": (
            "figures/figure_2_time_series_adaptive_seed_robustness.svg"
        ),
        "figure_2_pdf": (
            "figures/figure_2_time_series_adaptive_seed_robustness.pdf"
        ),
        "figure_3_svg": (
            "figures/figure_3_structured_multistep_seed_robustness.svg"
        ),
        "figure_3_pdf": (
            "figures/figure_3_structured_multistep_seed_robustness.pdf"
        ),
        "figure_4_svg": (
            "figures/figure_4_risk_control_ablations_seed_robustness.svg"
        ),
        "figure_4_pdf": (
            "figures/figure_4_risk_control_ablations_seed_robustness.pdf"
        ),
        "all_method_heatmap_svg": "figures/all_method_seed_heatmaps.svg",
        "all_method_heatmap_pdf": "figures/all_method_seed_heatmaps.pdf",
        "trace_split_curves_svg": "figures/trace_crc_seed_split_curves.svg",
        "trace_split_curves_pdf": "figures/trace_crc_seed_split_curves.pdf",
        "trace_split_heatmap_svg": "figures/trace_crc_seed_split_heatmaps.svg",
        "trace_split_heatmap_pdf": "figures/trace_crc_seed_split_heatmaps.pdf",
    }
)
with (robustness_dir / "manifest.json").open("w") as file:
    json.dump(manifest, file, indent=2)

print("Saved robustness figures to:", figure_dir)


In [ ]:
# =========================================================
# STANDALONE REPRODUCTION VALIDATION
# =========================================================

expected_method_seed_runs = len(INFERENCE_SEEDS) * len(METHOD_ORDER)
expected_trace_split_runs = (
    len(INFERENCE_SEEDS) * len(ROBUSTNESS_SPLIT_SEEDS)
)
risk_control_rows = [
    row
    for row in method_seed_rows
    if row["category"] == C_RISK
]

validation_checks = {
    "method_seed_run_count": (
        len(method_seed_rows) == expected_method_seed_runs
    ),
    "trace_split_run_count": (
        len(trace_split_rows) == expected_trace_split_runs
    ),
    "primary_calibration_size": (
        len(primary_cal_indices) == CALIBRATION_SIZE
    ),
    "primary_test_size": (
        len(primary_test_indices)
        == reference_shape[0] - CALIBRATION_SIZE
    ),
    "calibration_test_disjoint": (
        np.intersect1d(
            primary_cal_indices,
            primary_test_indices,
        ).size
        == 0
    ),
    "all_crc_selections_formal": all(
        bool(row["formal"])
        for row in risk_control_rows
    ),
    "no_crc_empirical_fallback": all(
        not bool(row["empirical_fallback_used"])
        for row in risk_control_rows
    ),
    "all_trace_robustness_selections_formal": all(
        bool(row["formal"])
        for row in trace_split_rows
    ),
    "test_data_excluded_from_tuning": (
        PARAMETER_SELECTION_ARTIFACT["test_data_used_for_tuning"]
        is False
    ),
}

if not all(validation_checks.values()):
    failed = [
        name
        for name, passed in validation_checks.items()
        if not passed
    ]
    raise RuntimeError(
        f"Standalone reproduction validation failed: {failed}"
    )

reproduction_validation = {
    "validation_type": "standalone_artifact",
    "all_checks_passed": True,
    "checks": validation_checks,
    "expected_method_seed_runs": expected_method_seed_runs,
    "observed_method_seed_runs": len(method_seed_rows),
    "expected_trace_split_runs": expected_trace_split_runs,
    "observed_trace_split_runs": len(trace_split_rows),
    "inference_seeds": [int(seed) for seed in INFERENCE_SEEDS],
    "split_seeds": [
        int(seed) for seed in ROBUSTNESS_SPLIT_SEEDS
    ],
    "empirical_fallback_allowed": False,
}

with (robustness_dir / "reproduction_validation.json").open(
    "w"
) as file:
    json.dump(reproduction_validation, file, indent=2)

manifest_path = robustness_dir / "manifest.json"
with manifest_path.open() as file:
    combined_manifest = json.load(file)
combined_manifest["files"]["reproduction_validation"] = (
    "reproduction_validation.json"
)
with manifest_path.open("w") as file:
    json.dump(combined_manifest, file, indent=2)

print("Standalone reproduction validation:", reproduction_validation)


## Interpretation

- Inference-seed robustness is compared fairly because every method uses the
  same predeclared random 300/700 trajectory-level split for each seed.
- TRACE-CRC sample-allocation robustness uses 10 predeclared random 300/700
  splits crossed with all five inference seeds, giving 50 evaluations.
- Within each TRACE calibration set, randomized calibration-index order creates
  disjoint profile, conformal-calibration, and LTT-validation subsets of
  sizes 45, 51, and 204.
- Baseline and TRACE hyperparameters remain fixed across inference and split
  seeds; only calibration-dependent quantities are recomputed.
- The 50 repeated-split evaluations reuse trajectories and are therefore
  descriptive rather than independent replications.
- LTT certification is per TRACE run at `delta=0.10`; aggregate robustness
  summaries do not provide a simultaneous certificate across runs.
